In [ ]:
import copy
import logging
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false" 
os.environ["WANDB_API_KEY"] = ""
import random
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple,Union
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
try:
    import wandb
except ImportError:
    wandb = None
import string
import re
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt

from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    import torch_xla.distributed.xla_multiprocessing as xmp
    TPU_AVAILABLE = True
except ImportError:
    TPU_AVAILABLE = False

try:
    from seqeval.metrics import f1_score, precision_score, recall_score
except ImportError as exc:  
    raise SystemExit("seqeval must be installed to run this script") from exc

try:
    import spacy
    from spacy.language import Language
except ImportError as exc:  # pragma: no cover - spaCy is required
    raise SystemExit("spaCy must be installed to run this script") from exc


LOGGER = logging.getLogger("pipeline")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Preprocessing & BIO Conversion
* **Custom spaCy Configuration:** Implements a pipeline component (`newline_boundary`) to force sentence segmentation on newlines.
* **Data Loading:** helper functions to read articles and gold-standard propaganda spans.
* **Feature Extraction:** Converts raw text into **BIO format**, enriching tokens with linguistic features (POS tags, NER types, and Dependency relations) via `create_bio_labeled` and `create_bio_unlabeled`.

In [ ]:
# Binary lexicon for Hitler-related keywords
HITLER_KEYWORDS = {
    "hitler", "nazi", "nazis", "nazism", "fascist", "fascism", "dictator",
    "totalitarian", "ss", "holocaust", "genocide", "reich", "gestapo",
}
@Language.component("newline_boundary")
def _newline_boundary_component(doc):
    """Custom spaCy component to force sentence segments on newlines."""
    for i, token in enumerate(doc[:-1]):
        # If the token contains a newline, the NEXT token is a start of a sentence
        if "\n" in token.text:
            doc[i + 1].is_sent_start = True
    return doc

def get_configured_spacy(model_name: str = "en_core_web_sm"):
    """Load and configure spaCy model with custom boundary rules."""
    try:
        nlp = spacy.load(model_name)
    except OSError:
        print("Falling back to blank English spaCy model; install %s for best accuracy", model_name)
        nlp = spacy.blank("en")
    
    # Add sentencizer if no parser/senter
    if not nlp.has_pipe("parser") and not nlp.has_pipe("senter") and "sentencizer" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")
    
    # Add newline boundary rule before the parser/senter to override
    if "newline_boundary" not in nlp.pipe_names:
        nlp.add_pipe("newline_boundary", before="parser" if nlp.has_pipe("parser") else None)
        
    return nlp

def load_data(data_folder: str, propaganda_techniques_file: str) -> Tuple[List[str], List[str], List[str]]:
    file_list = sorted(os.path.join(data_folder, name) for name in os.listdir(data_folder) if name.endswith(".txt"))
    articles_content: List[str] = []
    articles_id: List[str] = []
    for filename in file_list:
        with open(filename, "r", encoding="utf-8") as handle:
            articles_content.append(handle.read())
            articles_id.append(os.path.basename(filename).split(".")[0][7:])

    propaganda_techniques_names: List[str] = []
    if propaganda_techniques_file and os.path.exists(propaganda_techniques_file):
        with open(propaganda_techniques_file, "r", encoding="utf-8") as handle:
            propaganda_techniques_names = [line.rstrip("\n") for line in handle]

    return articles_content, articles_id, propaganda_techniques_names


def read_predictions_from_file(filename: str) -> Tuple[List[str], List[Tuple[int, int]]]:
    articles_id: List[str] = []
    gold_spans: List[Tuple[int, int]] = []
    with open(filename, "r", encoding="utf-8") as handle:
        for row in handle.readlines():
            article_id, gold_span_start, gold_span_end = row.rstrip("\n").split("\t")
            articles_id.append(article_id)
            gold_spans.append((int(gold_span_start), int(gold_span_end)))
    return articles_id, gold_spans


def group_spans_by_article_ids(span_list: Iterable[Tuple[str, Tuple[int, int]]]) -> Dict[str, List[Tuple[int, int]]]:
    data: Dict[str, List[Tuple[int, int]]] = {}
    for article_id, span in span_list:
        data.setdefault(article_id, []).append(span)
    return data


def token_label_from_spans(pos: int, spans: Sequence[Tuple[int, int]]) -> str:
    for start, end in spans:
        if start <= int(pos) < end:
            return "PROP"
    return "O"


def _iter_sentence_tokens(doc) -> Iterable[Sequence]:
    try:
        for sentence in doc.sents:
            yield list(sentence)
    except ValueError:
        yield list(doc)


def _clean_token_text(text: str) -> str:
    return text.strip().replace("\n", " ").replace("\t", " ")


def _has_hitler_keyword(word: str) -> int:
    """Return 1 if the token contains any Hitler-related keyword, else 0."""
    normalized = _clean_token_text(word).lower()
    return int(any(keyword in normalized for keyword in HITLER_KEYWORDS))


def create_bio_labeled(file_path: str, data: Sequence[Tuple[str, List[Tuple[int, int]]]], articles_content: Dict[str, str], nlp) -> None:
    with open(file_path, "w", encoding="utf-8") as handle:
        for article_id, spans in tqdm(data, desc=f"Writing {os.path.basename(file_path)}", leave=False):
            text = articles_content[article_id]
            if nlp is None:
                raise ValueError("spaCy model is required for extracting POS/NER features")
            
            # Process line-by-line 
            lines = text.splitlines()
            current_offset = 0
            wrote_sentence = False
            
            for line in lines:
                line_start = current_offset
                line_end = current_offset + len(line)
                
                if not line.strip():
                    current_offset = line_end + 1  # +1 for newline
                    continue
                
                # Process line with spaCy for tokenization and POS/NER
                doc = nlp(line)
                prev_label = "O"
                sentence_has_tokens = False
                
                for token in doc:
                    cleaned = _clean_token_text(token.text)
                    if not cleaned or repr(cleaned) in (repr("\ufeff"), repr("\u200f")):
                        prev_label = "O"
                        continue
                    
                    # Calculate token position in original article (not just line)
                    token_start = line_start + token.idx
                    label = token_label_from_spans(token_start, spans)
                    if label != "O":
                        label = ("I-" if prev_label != "O" else "B-") + "PROP"
                    
                    pos_tag = token.pos_ if token.pos_ else "X"
                    ent_type = token.ent_type_ if token.ent_type_ else "O"
                    dep_rel = token.dep_ if token.dep_ else "dep"  # Dependency relation

                    handle.write(f"{cleaned}\t{label}\t{pos_tag}\t{ent_type}\t{dep_rel}\n")
                    prev_label = label
                    sentence_has_tokens = True
                
                if sentence_has_tokens:
                    handle.write("\n")
                    wrote_sentence = True
                
                current_offset = line_end + 1  # +1 for newline character
            
            if not wrote_sentence:
                handle.write("\n")


def create_bio_unlabeled(file_path: str, articles_id: Sequence[str], articles_content: Sequence[str], nlp) -> None:
    with open(file_path, "w", encoding="utf-8") as handle:
        for article_id, text in tqdm(zip(articles_id, articles_content), total=len(articles_id), desc=f"Writing {os.path.basename(file_path)}", leave=False):
            if nlp is None:
                raise ValueError("spaCy model is required for extracting POS/NER features")
            
            # Use line-based splitting as requested
            lines = text.splitlines()
            wrote_sentence = False
            
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                
                # Treat each line as a sentence
                doc = nlp(line)
                
                sentence_has_tokens = False
                for token in doc:
                    cleaned = _clean_token_text(token.text)
                    if not cleaned or repr(cleaned) in (repr("\ufeff"), repr("\u200f")):
                        continue
                    
                    pos_tag = token.pos_ if token.pos_ else "X"
                    ent_type = token.ent_type_ if token.ent_type_ else "O"
                    dep_rel = token.dep_ if token.dep_ else "dep"  # Dependency relation

                    handle.write(f"{cleaned}\tO\t{pos_tag}\t{ent_type}\t{dep_rel}\n")
                    sentence_has_tokens = True
                
                if sentence_has_tokens:
                    handle.write("\n")
                    wrote_sentence = True
            
            if not wrote_sentence:
                # Ensure at least one newline for empty files/articles to define boundaries if needed? 
                handle.write("\n")


### Post-Processing & Submission Utils
This cell contains functions to handle the output of the model and convert it back to the competition format.

* **`TokenSpan` & `tokenize_like_bio`**: Ensures tokenization is consistent with the BIO format while preserving character offsets.
* **`labels_to_spans`**: Converts the predicted BIO tags (B-PROP, I-PROP) back into specific character start/end indices, merging adjacent spans.
* **`write_submission_file`**: Generates the final submission file (`article_id <tab> start <tab> end`).

In [ ]:
@dataclass
class TokenSpan:
    token: str
    start: int
    end: int


def tokenize_like_bio(text: str, nlp) -> List[TokenSpan]:
    spans: List[TokenSpan] = []
    if nlp is None:
        pos = 0
        for line in text.splitlines(True):
            for m in re.finditer(r"\S+", line):
                token_text = m.group()
                cleaned = token_text.strip().replace("\n", " ").replace("\t", " ")
                if len(cleaned) == 0:
                    continue
                if repr(cleaned) in (repr("\ufeff"), repr("\u200f")):
                    continue
                start = pos + m.start()
                end = pos + m.end()
                spans.append(TokenSpan(cleaned, start, end))
            pos += len(line)
        return spans
    for token in nlp(text):
        cleaned = token.text.strip().replace("\n", " ").replace("\t", " ")
        if len(cleaned) == 0:
            continue
        if repr(cleaned) in (repr("\ufeff"), repr("\u200f")):
            continue
        spans.append(TokenSpan(cleaned, int(token.idx), int(token.idx) + len(token.text)))
    return spans


def labels_to_spans(tokens: Sequence[TokenSpan], labels: Sequence[str]) -> List[Tuple[int, int]]:
    spans: List[Tuple[int, int]] = []
    active_start: Optional[int] = None
    active_end: Optional[int] = None
    for token, label in zip(tokens, labels):
        if label.startswith("B-"):
            if active_start is not None:
                spans.append((active_start, active_end if active_end is not None else token.end))
            active_start = token.start
            active_end = token.end
        elif label.startswith("I-") and active_start is not None:
            active_end = token.end
        else:
            if active_start is not None:
                spans.append((active_start, active_end if active_end is not None else token.end))
            active_start = None
            active_end = None
    if active_start is not None:
        spans.append((active_start, active_end if active_end is not None else active_start))
    merged: List[Tuple[int, int]] = []
    for start, end in spans:
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


def write_token_predictions(
    bio_source: str,
    predictions: Sequence[Sequence[str]],
    output_path: str,
) -> None:
    example_index = 0
    token_index = 0
    with open(bio_source, "r", encoding="utf-8") as reader, open(output_path, "w", encoding="utf-8") as writer:
        for line in reader:
            stripped = line.rstrip("\n")
            if stripped == "":
                writer.write("\n")
                if token_index != 0:
                    example_index += 1
                token_index = 0
                continue
            token = stripped.split("\t")[0]
            label = "O"
            if example_index < len(predictions) and token_index < len(predictions[example_index]):
                label = predictions[example_index][token_index]
            writer.write(f"{token}\t{label}\n")
            token_index += 1


def write_submission_file(spans: Dict[str, List[Tuple[int, int]]], output_path: str) -> None:
    with open(output_path, "w", encoding="utf-8") as handle:
        for article_id in sorted(spans.keys()):
            for start, end in sorted(spans[article_id]):
                handle.write(f"{article_id}\t{start}\t{end}\n")



### Dataset Loader & Feature Encoding
This cell handles the transformation of raw text data into tensor-ready features for BERT-based models:

* **`InputExample` & `InputFeatures`**: Dataclasses defining the structure of raw examples and their tokenized, numerical representations.
* **`read_examples_from_file`**: Parses the BIO-formatted dataset, extracting words, POS tags, NER tags, and dependency labels.
* **`convert_examples_to_features`**: The core function that tokenizes text (handling subword splitting), aligns labels (POS, NER, DEP, Technique) to tokens, applies padding/truncation, and generates `input_ids`, `attention_mask`, and `segment_ids`.
* **Label Mapping Utils**: Functions like `get_pos_ner_dep_maps` and `get_technique_labels` to create ID mappings for all auxiliary tasks.

In [ ]:
@dataclass
class InputExample:
    guid: str
    words: List[str]
    labels: List[str]
    pos_labels: Optional[List[str]] = None
    ner_labels: Optional[List[str]] = None
    dep_labels: Optional[List[str]] = None  # Dependency relation labels per token
    tech_labels: Optional[List[List[str]]] = None  # Technique labels per token (can be multi-label)


@dataclass
class InputFeatures:
    input_ids: List[int]
    attention_mask: List[int]
    token_type_ids: List[int]
    label_ids: List[int]
    pos_ids: Optional[List[int]] = None
    ner_ids: Optional[List[int]] = None
    dep_ids: Optional[List[int]] = None  # Dependency relation IDs per subtoken
    word_ids: Optional[List[int]] = None  # Maps each subtoken to its word index (-1 for special tokens)
    tech_multi_labels: Optional[List[List[int]]] = None  # [seq_len, num_techniques] multi-hot labels
    tech_label_mask: Optional[List[int]] = None  # [seq_len] 1 for supervised positions
    lexicon_features: Optional[List[List[int]]] = None  # Binary lexicon flags per subtoken


def read_examples_from_file(file_path: str, mode: str, expected_count: Optional[int] = None) -> List[InputExample]:
    guid_index = 1
    examples: List[InputExample] = []
    with open(file_path, encoding="utf-8") as handle:
        words: List[str] = []
        labels: List[str] = []
        current_pos: List[str] = []
        current_ner: List[str] = []
        current_dep: List[str] = []
        for line in handle:
            if line.startswith("-DOCSTART-") or line in ("", "\n"):
                if words:
                    examples.append(InputExample(
                        guid=f"{mode}-{guid_index}", 
                        words=words, 
                        labels=labels, 
                        pos_labels=current_pos, 
                        ner_labels=current_ner,
                        dep_labels=current_dep
                    ))
                    guid_index += 1
                    words = []
                    labels = []
                    current_pos = []
                    current_ner = []
                    current_dep = []
            else:
                splits = line.split("\t")
                words.append(splits[0])
                if len(splits) > 1:
                    labels.append(splits[1].rstrip("\n"))
                else:
                    labels.append("O")
                
                # Check for POS column (column 2)
                if len(splits) > 2:
                    current_pos.append(splits[2])
                else:
                    current_pos.append("X")  # Default/Fallback
                
                # Check for NER column (column 3)
                if len(splits) > 3:
                    current_ner.append(splits[3].rstrip("\n") if len(splits) == 4 else splits[3])
                else:
                    current_ner.append("O")  # Default/Fallback
                
                # Check for DEP column (column 4)
                if len(splits) > 4:
                    current_dep.append(splits[4].rstrip("\n"))
                else:
                    current_dep.append("dep")  # Default/Fallback
        if words:
            examples.append(InputExample(
                guid=f"{mode}-{guid_index}", 
                words=words, 
                labels=labels, 
                pos_labels=current_pos, 
                ner_labels=current_ner,
                dep_labels=current_dep
            ))
    if expected_count is not None and len(examples) != expected_count:
        raise ValueError(f"Expected {expected_count} {mode} examples, found {len(examples)} in {file_path}")
    return examples


def convert_examples_to_features(
    examples: Sequence[InputExample],
    label_list: Sequence[str],
    max_seq_length: int,
    tokenizer,
    pad_token_label_id: int,
    pos_label_map: Optional[Dict[str, int]] = None,
    ner_label_map: Optional[Dict[str, int]] = None,
    dep_label_map: Optional[Dict[str, int]] = None,
    tech_label_map: Optional[Dict[str, int]] = None,
) -> List[InputFeatures]:
    """
    Convert InputExamples to InputFeatures with subword-to-word mapping.
    
    Args:
        pos_label_map: Mapping from POS tag to ID
        ner_label_map: Mapping from NER tag to ID
        dep_label_map: Mapping from dependency relation to ID
        tech_label_map: Optional mapping from technique names to IDs
    
    Returns:
        List of InputFeatures with word_ids for mean pooling
    """
    label_map = {label: i for i, label in enumerate(label_list)}
    sep_token = tokenizer.sep_token
    cls_token = tokenizer.cls_token
    pad_token = tokenizer.pad_token_id

    # Fallback to empty if None
    pos_map = pos_label_map if pos_label_map else {}
    ner_map = ner_label_map if ner_label_map else {}
    dep_map = dep_label_map if dep_label_map else {}
    tech_map = tech_label_map if tech_label_map else {}
    num_techniques = len(tech_map)

    features: List[InputFeatures] = []
    for example in examples:
        tokens: List[str] = []
        label_ids: List[int] = []
        pos_ids: List[int] = []
        ner_ids: List[int] = []
        dep_ids: List[int] = []  # Dependency relation IDs
        word_ids: List[int] = []  # Maps each subtoken to word index
        tech_multi_labels: List[List[int]] = []  # Multi-label technique targets
        tech_label_mask: List[int] = []  # Mask for supervised positions
        lexicon_features: List[List[int]] = []  # Binary lexicon feature per subtoken

        # Handle missing optional labels
        ex_pos_labels = example.pos_labels if example.pos_labels else ["X"] * len(example.words)
        ex_ner_labels = example.ner_labels if example.ner_labels else ["O"] * len(example.words)
        ex_dep_labels = example.dep_labels if example.dep_labels else ["dep"] * len(example.words)
        ex_tech_labels = example.tech_labels if example.tech_labels else [[] for _ in range(len(example.words))]

        for word_idx, (word, label, pos, ner, dep, tech_list) in enumerate(
            zip(example.words, example.labels, ex_pos_labels, ex_ner_labels, ex_dep_labels, ex_tech_labels)
        ):
            word_tokens = tokenizer.tokenize(word)
            if not word_tokens:
                word_tokens = [tokenizer.unk_token]
            tokens.extend(word_tokens)
            lex_flag = _has_hitler_keyword(word)
            
            # Align labels: First subtoken gets label, others get pad
            label_ids.extend([label_map[label]] + [pad_token_label_id] * (len(word_tokens) - 1))
            
            # Align POS: First subtoken gets POS, others get 0 (padding for embedding)
            p_id = pos_map.get(pos, 0)
            pos_ids.extend([p_id] + [0] * (len(word_tokens) - 1))

            # Align NER: First subtoken gets NER, others get 0
            n_id = ner_map.get(ner, 0)
            ner_ids.extend([n_id] + [0] * (len(word_tokens) - 1))
            
            # Align DEP: First subtoken gets DEP, others get 0
            d_id = dep_map.get(dep, 0)
            dep_ids.extend([d_id] + [0] * (len(word_tokens) - 1))
            
            # word_ids - ALL subtokens of a word get the same word index
            word_ids.extend([word_idx] * len(word_tokens))

            # NEW: multi-label technique targets
            # First subtoken is supervised, remaining subtokens are masked out.
            multi = [0] * num_techniques
            if isinstance(tech_list, str):
                tech_list = [tech_list] if tech_list and tech_list != "O" else []
            for tech in (tech_list or []):
                idx = tech_map.get(tech)
                if idx is not None:
                    multi[idx] = 1
            tech_multi_labels.extend([multi] + [[0] * num_techniques] * (len(word_tokens) - 1))
            tech_label_mask.extend([1] + [0] * (len(word_tokens) - 1))

            # NEW: lexicon_features - All subtokens inherit the binary flag
            lexicon_features.extend([[lex_flag]] * len(word_tokens))

        special_tokens_count = 2
        if len(tokens) > max_seq_length - special_tokens_count:
            tokens = tokens[: max_seq_length - special_tokens_count]
            label_ids = label_ids[: max_seq_length - special_tokens_count]
            pos_ids = pos_ids[: max_seq_length - special_tokens_count]
            ner_ids = ner_ids[: max_seq_length - special_tokens_count]
            dep_ids = dep_ids[: max_seq_length - special_tokens_count]
            word_ids = word_ids[: max_seq_length - special_tokens_count]
            tech_multi_labels = tech_multi_labels[: max_seq_length - special_tokens_count]
            tech_label_mask = tech_label_mask[: max_seq_length - special_tokens_count]
            lexicon_features = lexicon_features[: max_seq_length - special_tokens_count]

        # Add SEP token
        tokens += [sep_token]
        label_ids += [pad_token_label_id]
        pos_ids += [0]
        ner_ids += [0]
        dep_ids += [0]
        word_ids += [-1]  # -1 for special tokens
        tech_multi_labels += [[0] * num_techniques]
        tech_label_mask += [0]
        lexicon_features += [[0]]
        segment_ids = [0] * len(tokens)

        # Add CLS token at beginning
        tokens = [cls_token] + tokens
        label_ids = [pad_token_label_id] + label_ids
        pos_ids = [0] + pos_ids
        ner_ids = [0] + ner_ids
        dep_ids = [0] + dep_ids
        word_ids = [-1] + word_ids  # -1 for CLS
        tech_multi_labels = [[0] * num_techniques] + tech_multi_labels
        tech_label_mask = [0] + tech_label_mask
        lexicon_features = [[0]] + lexicon_features
        segment_ids = [0] + segment_ids

        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)

        padding_length = max_seq_length - len(input_ids)
        input_ids += [pad_token] * padding_length
        attention_mask += [0] * padding_length
        segment_ids += [0] * padding_length
        label_ids += [pad_token_label_id] * padding_length
        pos_ids += [0] * padding_length
        ner_ids += [0] * padding_length
        dep_ids += [0] * padding_length
        word_ids += [-1] * padding_length  # -1 for padding
        tech_multi_labels += [[0] * num_techniques] * padding_length
        tech_label_mask += [0] * padding_length
        lexicon_features += [[0]] * padding_length

        features.append(
            InputFeatures(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=segment_ids,
                label_ids=label_ids,
                pos_ids=pos_ids,
                ner_ids=ner_ids,
                dep_ids=dep_ids,
                word_ids=word_ids,
                tech_multi_labels=tech_multi_labels,
                tech_label_mask=tech_label_mask,
                lexicon_features=lexicon_features,
            )
        )
    return features


def get_labels(path: Optional[str] = None) -> List[str]:
    if path:
        with open(path, "r", encoding="utf-8") as handle:
            labels = handle.read().splitlines()
        if "O" not in labels:
            labels = ["O"] + labels
        return labels
    return ["O", "B-PROP", "I-PROP"]


def get_pos_ner_dep_maps(examples: List[InputExample]) -> Tuple[Dict[str, int], Dict[str, int], Dict[str, int]]:
    """Build POS, NER, and DEP label maps from training examples.
    
    Returns:
        Tuple of (pos_map, ner_map, dep_map) where each maps label string to int ID.
        Index 0 is reserved for padding.
    """
    pos_types = set()
    ner_types = set()
    dep_types = set()
    for ex in examples:
        if ex.pos_labels:
            pos_types.update(ex.pos_labels)
        if ex.ner_labels:
            ner_types.update(ex.ner_labels)
        if ex.dep_labels:
            dep_types.update(ex.dep_labels)
    
    # 0 is reserved for padding, so start maps from 1
    sorted_pos = sorted(list(pos_types))
    sorted_ner = sorted(list(ner_types))
    sorted_dep = sorted(list(dep_types))
    
    pos_map = {label: i + 1 for i, label in enumerate(sorted_pos)}
    ner_map = {label: i + 1 for i, label in enumerate(sorted_ner)}
    dep_map = {label: i + 1 for i, label in enumerate(sorted_dep)}
    
    return pos_map, ner_map, dep_map


# Keep backwards compatibility
def get_pos_ner_maps(examples: List[InputExample]) -> Tuple[Dict[str, int], Dict[str, int]]:
    """Backwards compatible: returns only pos_map and ner_map."""
    pos_map, ner_map, _ = get_pos_ner_dep_maps(examples)
    return pos_map, ner_map


def get_technique_labels(path: Optional[str] = None) -> List[str]:
    """
    Load propaganda technique labels from file.
    
    Args:
        path: Path to propaganda-techniques-names.txt (14 techniques)
    
    Returns:
        List of technique labels with "O" (non-propaganda) at index 0
    """
    if path:
        with open(path, "r", encoding="utf-8") as handle:
            techniques = [line.strip() for line in handle if line.strip()]
        return ["O"] + techniques
    # Default 14 propaganda techniques from SemEval-2020
    return [
        "O",
        "Appeal_to_Authority",
        "Appeal_to_fear-prejudice",
        "Bandwagon,Reductio_ad_hitlerum",
        "Black-and-White_Fallacy",
        "Causal_Oversimplification",
        "Doubt",
        "Exaggeration,Minimisation",
        "Flag-Waving",
        "Loaded_Language",
        "Name_Calling,Labeling",
        "Repetition",
        "Slogans",
        "Thought-terminating_Cliches",
        "Whataboutism,Straw_Men,Red_Herring",
    ]


def read_technique_spans(filename: str) -> Dict[str, List[Tuple[int, int, str]]]:
    """
    Read technique classification spans from task2 labels file.
    
    Format: article_id<TAB>technique<TAB>start<TAB>end
    
    Returns:
        Dict mapping article_id to list of (start, end, technique) tuples
    """
    spans: Dict[str, List[Tuple[int, int, str]]] = {}
    with open(filename, "r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 4:
                article_id = parts[0]
                technique = parts[1]
                start = int(parts[2])
                end = int(parts[3])
                spans.setdefault(article_id, []).append((start, end, technique))
    return spans


def get_techniques_for_token(
    token_start: int,
    token_end: int,
    technique_spans: List[Tuple[int, int, str]],
) -> List[str]:
    """
    Get the technique labels for a token based on overlapping spans.

    Task 2 annotations can overlap, meaning a token may belong to multiple techniques.
    We return all overlapping techniques (unique, stable order).
    
    Args:
        token_start: Character start position of the token
        token_end: Character end position of the token
        technique_spans: List of (start, end, technique) tuples
    
    Returns:
        Technique name if overlapping, otherwise "O"
    """
    overlaps: List[str] = []
    for span_start, span_end, technique in technique_spans:
        # Check for any overlap
        if token_start < span_end and token_end > span_start:
            overlaps.append(technique)

    if not overlaps:
        return []

    # Unique techniques overlapped by this token
    return list(dict.fromkeys(overlaps))


def get_technique_map(technique_labels: List[str]) -> Dict[str, int]:
    """Create a mapping from technique names to IDs."""
    return {label: i for i, label in enumerate(technique_labels)}


def get_multilabel_technique_map(technique_labels: List[str]) -> Dict[str, int]:
    """Map technique name -> multi-label index (excludes 'O' if present)."""
    labels = list(technique_labels)
    if labels and labels[0] == "O":
        labels = labels[1:]
    return {label: i for i, label in enumerate(labels)}


def attach_technique_labels_to_examples(
    examples: List[InputExample],
    *,
    article_ids: Sequence[str],
    token_cache: Dict[str, List["TokenSpan"]],
    technique_spans_by_article: Dict[str, List[Tuple[int, int, str]]],
) -> None:
    """Populate `example.tech_labels` by aligning BIO tokens to `token_cache` offsets.

    This assumes the BIO examples are written in the same article order as `article_ids`,
    and that the concatenation of example token counts matches the token_cache token counts
    per article (as used elsewhere in this script).
    """

    def _norm_token(text: str) -> str:
        return _clean_token_text(text)

    example_index = 0
    for article_id in article_ids:
        tokens = token_cache.get(article_id, [])
        technique_spans = technique_spans_by_article.get(article_id, [])
        token_index = 0

        while token_index < len(tokens) and example_index < len(examples):
            example = examples[example_index]
            tech_labels: List[List[str]] = []

            remaining = len(tokens) - token_index
            # If we are about to overrun the article token list, still assign best-effort labels
            # and then stop consuming this article.
            overrun = len(example.words) > remaining

            for word in example.words:
                if token_index >= len(tokens):
                    tech_labels.append([])
                    continue

                token_span = tokens[token_index]

                # Best-effort resync if token text mismatches.
                if _norm_token(token_span.token) != _norm_token(word):
                    lookahead_limit = min(len(tokens), token_index + 6)
                    found_at = None
                    for j in range(token_index + 1, lookahead_limit):
                        if _norm_token(tokens[j].token) == _norm_token(word):
                            found_at = j
                            break
                    if found_at is not None:
                        token_index = found_at
                        token_span = tokens[token_index]

                tech_labels.append(get_techniques_for_token(token_span.start, token_span.end, technique_spans))
                token_index += 1

            example.tech_labels = tech_labels
            example_index += 1

            if overrun:
                break

        if token_index != len(tokens):
            print(
                f"[WARN] Technique-label alignment: article {article_id} token mismatch: "
                f"consumed={token_index} cache_len={len(tokens)}"
            )

    if example_index != len(examples):
        print(
            f"[WARN] Technique-label alignment: unused BIO examples: {len(examples) - example_index}"
        )

### CRF Layer & Viterbi Decoding
**Conditional Random Field (CRF)** 

* **`ConditionalRandomField`**: The main `nn.Module` class that computes the log-likelihood of tag sequences during training and manages transition parameters.
* **`viterbi_decode`**: Implements the **Viterbi algorithm** to find the most probable sequence of tags during inference.
* **`allowed_transitions`**: Generates a constraint mask (specifically for **BIO tagging**) to enforce valid state transitions (e.g., preventing a transition from "O" directly to "I-PROP").

In [ ]:
VITERBI_DECODING = Tuple[List[int], float]


def logsumexp(tensor: torch.Tensor, dim: int = -1, keepdim: bool = False) -> torch.Tensor:
    max_score, _ = tensor.max(dim, keepdim=keepdim)
    if keepdim:
        stable_vec = tensor - max_score
    else:
        stable_vec = tensor - max_score.unsqueeze(dim)
    return max_score + (stable_vec.exp().sum(dim, keepdim=keepdim)).log()


class ConditionalRandomField(torch.nn.Module):
    def __init__(
        self,
        num_tags: int,
        constraints: Optional[List[Tuple[int, int]]] = None,
        include_start_end_transitions: bool = True,
    ) -> None:
        super().__init__()
        self.num_tags = num_tags
        self.transitions = torch.nn.Parameter(torch.empty(num_tags, num_tags))
        if constraints is None:
            constraint_mask = torch.ones(num_tags + 2, num_tags + 2)
        else:
            constraint_mask = torch.zeros(num_tags + 2, num_tags + 2)
            for i, j in constraints:
                constraint_mask[i, j] = 1.0
        self._constraint_mask = torch.nn.Parameter(constraint_mask, requires_grad=False)
        self.include_start_end_transitions = include_start_end_transitions
        if include_start_end_transitions:
            self.start_transitions = torch.nn.Parameter(torch.empty(num_tags))
            self.end_transitions = torch.nn.Parameter(torch.empty(num_tags))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        torch.nn.init.xavier_normal_(self.transitions)
        if self.include_start_end_transitions:
            torch.nn.init.normal_(self.start_transitions)
            torch.nn.init.normal_(self.end_transitions)

    def _input_likelihood(self, logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, num_tags = logits.size()
        mask = mask.float().transpose(0, 1).contiguous()
        logits = logits.transpose(0, 1).contiguous()
        if self.include_start_end_transitions:
            alpha = self.start_transitions.view(1, num_tags) + logits[0]
        else:
            alpha = logits[0]
        for index in range(1, sequence_length):
            emit_scores = logits[index].view(batch_size, 1, num_tags)
            transition_scores = self.transitions.view(1, num_tags, num_tags)
            broadcast_alpha = alpha.view(batch_size, num_tags, 1)
            inner = broadcast_alpha + emit_scores + transition_scores
            alpha = logsumexp(inner, 1) * mask[index].view(batch_size, 1) + alpha * (1 - mask[index]).view(batch_size, 1)
        if self.include_start_end_transitions:
            stops = alpha + self.end_transitions.view(1, num_tags)
        else:
            stops = alpha
        return logsumexp(stops)

    def _joint_likelihood(self, logits: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = logits.data.shape
        logits = logits.transpose(0, 1).contiguous()
        mask = mask.float().transpose(0, 1).contiguous()
        tags_ = tags.transpose(0, 1).contiguous()
        if self.include_start_end_transitions:
            score = self.start_transitions.index_select(0, tags_[0])
        else:
            score = 0.0
        for i in range(sequence_length - 1):
            current_tag, next_tag = tags_[i], tags_[i + 1]
            transition_score = self.transitions[current_tag.view(-1), next_tag.view(-1)]
            emit_score = logits[i].gather(1, current_tag.view(batch_size, 1)).squeeze(1)
            score = score + transition_score * mask[i + 1] + emit_score * mask[i]
        last_tag_index = mask.sum(0).long() - 1
        last_tags = tags_.gather(0, last_tag_index.view(1, batch_size)).squeeze(0)
        if self.include_start_end_transitions:
            last_transition_score = self.end_transitions.index_select(0, last_tags)
        else:
            last_transition_score = 0.0
        last_inputs = logits[-1]
        last_input_score = last_inputs.gather(1, last_tags.view(-1, 1)).squeeze()
        score = score + last_transition_score + last_input_score * mask[-1]
        return score

    def forward(self, inputs: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        if mask is None:
            mask = torch.ones_like(tags, dtype=torch.long)
        log_denominator = self._input_likelihood(inputs, mask)
        log_numerator = self._joint_likelihood(inputs, tags, mask)
        return torch.sum(log_numerator - log_denominator)

    def viterbi_tags(
        self, logits: torch.Tensor, mask: torch.Tensor, top_k: int = 1
    ) -> List[List[VITERBI_DECODING]]:
        if top_k is None or top_k < 1:
            raise ValueError("top_k must be >= 1")
        flatten_output = top_k == 1
        _, max_seq_length, num_tags = logits.size()
        logits, mask = logits.data, mask.data
        start_tag = num_tags
        end_tag = num_tags + 1
        transitions = torch.full((num_tags + 2, num_tags + 2), -10000.0)
        constrained_transitions = self.transitions * self._constraint_mask[:num_tags, :num_tags] + -10000.0 * (
            1 - self._constraint_mask[:num_tags, :num_tags]
        )
        transitions[:num_tags, :num_tags] = constrained_transitions.data
        if self.include_start_end_transitions:
            transitions[start_tag, :num_tags] = self.start_transitions * self._constraint_mask[start_tag, :num_tags]
            transitions[:num_tags, end_tag] = self.end_transitions * self._constraint_mask[:num_tags, end_tag]
        else:
            transitions[start_tag, :num_tags] = -10000.0 * (1 - self._constraint_mask[start_tag, :num_tags])
            transitions[:num_tags, end_tag] = -10000.0 * (1 - self._constraint_mask[:num_tags, end_tag])

        best_paths: List[List[VITERBI_DECODING]] = []
        tag_sequence = torch.zeros(max_seq_length + 2, num_tags + 2)
        for prediction, prediction_mask in zip(logits, mask):
            sequence_length = int(torch.sum(prediction_mask))
            tag_sequence.fill_(-10000.0)
            tag_sequence[0, start_tag] = 0.0
            tag_sequence[1 : sequence_length + 1, :num_tags] = prediction[:sequence_length]
            tag_sequence[sequence_length + 1, end_tag] = 0.0
            paths, scores = viterbi_decode(tag_sequence[: sequence_length + 2], transitions, top_k=top_k)
            cleaned_paths: List[VITERBI_DECODING] = []
            for path, score in zip(paths, scores):
                cleaned_paths.append((path[1:-1], float(score.item())))
            best_paths.append(cleaned_paths)
        if flatten_output:
            return [[paths[0]] for paths in best_paths]
        return best_paths


def is_transition_allowed(constraint_type: str, from_tag: str, from_entity: str, to_tag: str, to_entity: str) -> bool:
    if to_tag == "START" or from_tag == "END":
        return False
    if constraint_type == "BIO":
        if from_tag == "START":
            return to_tag in ("O", "B")
        if to_tag == "END":
            return from_tag in ("O", "B", "I")
        if to_tag in ("O", "B"):
            return True
        return to_tag == "I" and from_tag in ("B", "I") and from_entity == to_entity
    raise ValueError(f"Unsupported constraint type: {constraint_type}")


def allowed_transitions(constraint_type: str, labels: Dict[int, str]) -> List[Tuple[int, int]]:
    num_labels = len(labels)
    start_tag = num_labels
    end_tag = num_labels + 1
    labels_with_boundaries = list(labels.items()) + [(start_tag, "START"), (end_tag, "END")]
    allowed: List[Tuple[int, int]] = []
    for from_label_index, from_label in labels_with_boundaries:
        from_tag = from_label if from_label in ("START", "END") else from_label[0]
        from_entity = "" if from_label in ("START", "END") else from_label[1:]
        for to_label_index, to_label in labels_with_boundaries:
            to_tag = to_label if to_label in ("START", "END") else to_label[0]
            to_entity = "" if to_label in ("START", "END") else to_label[1:]
            if is_transition_allowed(constraint_type, from_tag, from_entity, to_tag, to_entity):
                allowed.append((from_label_index, to_label_index))
    return allowed


def viterbi_decode(
    tag_sequence: torch.Tensor,
    transition_matrix: torch.Tensor,
    top_k: int = 1,
) -> Tuple[List[List[int]], torch.Tensor]:
    sequence_length, num_tags = tag_sequence.size()
    path_scores: List[torch.Tensor] = []
    path_indices: List[torch.Tensor] = []
    path_scores.append(tag_sequence[0].unsqueeze(0))
    for timestep in range(1, sequence_length):
        summed_potentials = path_scores[timestep - 1].unsqueeze(2) + transition_matrix
        summed_potentials = summed_potentials.view(-1, num_tags)
        scores, paths = torch.topk(summed_potentials, k=min(summed_potentials.size(0), top_k), dim=0)
        path_scores.append(tag_sequence[timestep].unsqueeze(0) + scores)
        path_indices.append(paths.squeeze())
    path_scores_v = path_scores[-1].view(-1)
    max_k = min(path_scores_v.size(0), top_k)
    viterbi_scores, best_paths = torch.topk(path_scores_v, k=max_k, dim=0)
    viterbi_paths: List[List[int]] = []
    for i in range(max_k):
        viterbi_path = [int(best_paths[i])]
        for backward_timestep in reversed(path_indices):
            viterbi_path.append(int(backward_timestep.view(-1)[viterbi_path[-1]]))
        viterbi_path.reverse()
        viterbi_paths.append(viterbi_path)
    return viterbi_paths, viterbi_scores


### Scalar Mixture Layer
This cell defines the `ScalarMix` module, which computes a **learnable weighted sum** of multiple tensors 
* **Mechanism:** It applies a softmax to learnable scalar weights to determine the importance of each input tensor, then scales the combined result by a trainable parameter $\gamma$.
* **Formula:** $mixture = \gamma \times \sum_{k} (\text{softmax}(w_k) \times \text{tensor}_k)$

In [ ]:
class ScalarMix(nn.Module):
    r"""
    Computes a parameterised scalar mixture of N tensors, $mixture = \gamma * \sum(s_k * tensor_k)$
    where $s = \text{softmax}(w)$, with $w$ and $\gamma$ being the trainable parameters.
    """

    def __init__(
        self,
        mixture_size: int,
        do_layer_norm: bool = False,
        initial_scalar_parameters: Optional[Sequence[float]] = None,
        trainable: bool = True,
    ) -> None:
        super().__init__()
        self.mixture_size = mixture_size
        self.do_layer_norm = do_layer_norm

        if initial_scalar_parameters is None:
            initial_scalar_parameters = [0.0] * mixture_size
        elif len(initial_scalar_parameters) != mixture_size:
            raise ValueError(
                f"Length of initial_scalar_parameters {len(initial_scalar_parameters)} "
                f"differs from mixture_size {mixture_size}"
            )

        self.scalar_parameters = nn.Parameter(
            torch.tensor(initial_scalar_parameters, dtype=torch.float32), requires_grad=trainable
        )
        self.gamma = nn.Parameter(torch.tensor(1.0, dtype=torch.float32), requires_grad=trainable)

        if self.do_layer_norm:
            self.layer_norms = nn.ModuleList([nn.LayerNorm(normalized_shape=None) for _ in range(mixture_size)]) # Normalized shape inferred at runtime? No, LayerNorm needs shape. 
            pass

    def forward(self, tensors: Sequence[torch.Tensor], mask: Optional[torch.BoolTensor] = None) -> torch.Tensor:
        if len(tensors) != self.mixture_size:
            raise ValueError(
                f"ScalarMix expects {self.mixture_size} tensors but got {len(tensors)}"
            )

        normed_weights = torch.softmax(self.scalar_parameters, dim=0)
        
        # Weighted sum
        # tensors: List of [Batch, Time, Dim]
        # We want to compute sum_k (w_k * tensor_k)
        
        # Stack to [Mixture, Batch, Time, Dim] for easier broadcasting if memory allows, 
        # or just loop sum. Loop sum saves memory peak.
        
        mixture = torch.zeros_like(tensors[0])
        for i, tensor in enumerate(tensors):
            mixture = mixture + normed_weights[i] * tensor
            
        return self.gamma * mixture


### Auxiliary Task: Sentence Classification
This cell implements the `SentenceAuxiliaryTask`, a module designed to classify entire sentences as "propaganda" or "non-propaganda" before fine-grained token labeling.

* **Purpose:** Acts as a **hard gate**. If a sentence is classified as non-propaganda (gate=0), the model can skip or zero-out token-level predictions, reducing false positives.
* **Architecture:**
    * **Separate Representation:** Uses its own `ScalarMix` to combine PLM layers, distinct from the token-level task.
    * **BiLSTM Encoder:** Processes the sentence embeddings to capture context.
    * **Structural Features:** Concatenates **sentence length** and **sentence position** (upper/middle/lower part of article) to the BiLSTM output.
* **Gating Logic:**
    * **Training:** Uses ground truth labels to gate the loss (only calculate token loss if the sentence is actually propaganda).
    * **Inference:** Uses predicted probability (> 0.5) to activate the token classifier.

In [ ]:
class SentenceAuxiliaryTask(nn.Module):
    """
    Sentence-level auxiliary task for propaganda detection.
    
    Uses separate PLM layer attention and BiLSTM from the token-level task.
    Incorporates structural features (sentence length, position in article).
    Implements hard gating: gate=0 for non-propaganda, gate=1 for propaganda sentences.
    
    Loss behavior:
    - Non-propaganda sentences: only sentence classification loss
    - Propaganda sentences: sentence loss + lambda * token-level BIO loss
    """
    
    def __init__(
        self,
        hidden_size: int,
        num_plm_layers: int,
        pos_embedding_dim: int = 0,
        ner_embedding_dim: int = 0,
        dep_embedding_dim: int = 0,
        lexicon_feature_dim: int = 0,
        rnn_layers: int = 1,
        rnn_hidden_size: int = 256,
        rnn_dropout: float = 0.1,
        output_dropout: float = 0.1,
        structural_feature_dim: int = 4,  # length + upper/middle/lower position
    ) -> None:
        super().__init__()
        
        # Separate scalar mix for sentence-level PLM attention
        self.scalar_mix_sent = ScalarMix(num_plm_layers)
        
        # Input dim: PLM hidden + optional token features
        self.embedding_dim = hidden_size + pos_embedding_dim + ner_embedding_dim + dep_embedding_dim + lexicon_feature_dim
        
        # Sentence-level BiLSTM
        self.rnn_hidden_size = rnn_hidden_size if rnn_hidden_size > 0 else self.embedding_dim // 2
        self.sent_lstm = nn.LSTM(
            self.embedding_dim,
            self.rnn_hidden_size,
            num_layers=rnn_layers,
            bidirectional=True,
            batch_first=True,
            dropout=rnn_dropout if rnn_layers > 1 else 0.0,
        )
        
        # BiLSTM output dim (bidirectional)
        lstm_output_dim = self.rnn_hidden_size * 2
        
        # Structural feature dimensions: 
        # - normalized sentence length (1)
        # - position flags: upper/middle/lower (3)
        self.structural_feature_dim = structural_feature_dim
        
        # FFN for sentence classification
        # Input: BiLSTM BoS hidden + structural features
        ffn_input_dim = lstm_output_dim + structural_feature_dim
        self.sent_ffn = nn.Sequential(
            nn.Linear(ffn_input_dim, lstm_output_dim),
            nn.ReLU(),
            nn.Dropout(output_dropout),
        )
        
        # Final sentence classifier
        self.sent_classifier = nn.Linear(lstm_output_dim, 1)
        
        self.dropout = nn.Dropout(output_dropout)
    
    def forward(
        self,
        plm_hidden_states: Tuple[torch.Tensor, ...],  # All PLM layer outputs
        pos_embeddings: Optional[torch.Tensor] = None,
        ner_embeddings: Optional[torch.Tensor] = None,
        dep_embeddings: Optional[torch.Tensor] = None,
        lexicon_features: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        pad_token_label_id: int = -100,
        o_label_index: int = 0,
        sentence_lengths: Optional[torch.Tensor] = None,  # [B]
        sentence_positions: Optional[torch.Tensor] = None,  # [B, 3] one-hot: upper/middle/lower
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass for sentence auxiliary task.
        
        Returns:
            dict with keys:
            - sentence_logits: [B] raw logits for sentence classification
            - sentence_probs: [B] probability that sentence contains propaganda
            - sentence_labels: [B] ground truth (1 if contains propaganda, 0 otherwise)
            - gates: [B, T] hard gates (0 or 1) for gating token-level loss
            - hard_gates: [B] hard gate per sentence (0 or 1)
        """
        B = plm_hidden_states[0].shape[0]
        T = plm_hidden_states[0].shape[1]
        device = plm_hidden_states[0].device
        
        if attention_mask is None:
            attention_mask = torch.ones(B, T, device=device, dtype=torch.long)
        
        # Get sentence-level PLM representation via separate scalar mix
        sent_plm_output = self.scalar_mix_sent(plm_hidden_states)
        sent_plm_output = self.dropout(sent_plm_output)
        
        # Concatenate POS/NER embeddings if available
        features_to_concat = [sent_plm_output]
        if pos_embeddings is not None:
            features_to_concat.append(pos_embeddings)
        if ner_embeddings is not None:
            features_to_concat.append(ner_embeddings)
        if dep_embeddings is not None:
            features_to_concat.append(dep_embeddings)
        if lexicon_features is not None:
            features_to_concat.append(lexicon_features)
        
        if len(features_to_concat) > 1:
            sent_input = torch.cat(features_to_concat, dim=-1)
        else:
            sent_input = sent_plm_output
        
        # Pass through sentence BiLSTM
        sent_lstm_out, _ = self.sent_lstm(sent_input)
        
        # Use BoS token (first token, typically [CLS]) for sentence representation
        bos_hidden = sent_lstm_out[:, 0, :]  # [B, lstm_output_dim]
        
        # Construct structural features
        if sentence_lengths is not None:
            # Normalize sentence length (divide by 512 as typical max)
            norm_lengths = sentence_lengths.float().unsqueeze(-1) / 512.0  # [B, 1]
        else:
            # Use attention mask sum as proxy for sentence length
            norm_lengths = attention_mask.sum(dim=1, keepdim=True).float() / 512.0  # [B, 1]
        
        if sentence_positions is not None:
            # sentence_positions should be [B, 3] one-hot for upper/middle/lower
            pos_features = sentence_positions.float()  # [B, 3]
        else:
            # Default to middle position if not provided
            pos_features = torch.zeros(B, 3, device=device)
            pos_features[:, 1] = 1.0  # middle
        
        structural_features = torch.cat([norm_lengths, pos_features], dim=-1)  # [B, 4]
        
        # Concatenate BoS hidden with structural features
        combined = torch.cat([bos_hidden, structural_features], dim=-1)  # [B, lstm_output_dim + 4]
        
        # FFN + classifier
        ffn_out = self.sent_ffn(combined)
        sentence_logits = self.sent_classifier(ffn_out).squeeze(-1)  # [B]
        sentence_probs = torch.sigmoid(sentence_logits)
        
        # Compute ground truth sentence labels from token labels
        sentence_labels = None
        if labels is not None:
            # A sentence contains propaganda if any token has a non-O, non-padding label
            valid_mask = labels != pad_token_label_id
            non_o_mask = (labels != o_label_index) & valid_mask
            sentence_labels = non_o_mask.any(dim=1).float()  # [B]
        
        # Hard gating: gate = 1 if sentence contains propaganda, 0 otherwise
        # During training: use ground truth labels for gating
        # During inference: use predicted probabilities (threshold 0.5)
        if sentence_labels is not None:
            # Training: use ground truth
            hard_gates = sentence_labels  # [B]
        else:
            # Inference: use predictions
            hard_gates = (sentence_probs > 0.5).float()  # [B]
        
        # Expand gates to token dimension for masking token-level loss
        gates = hard_gates.unsqueeze(1).expand(B, T)  # [B, T]
        
        return {
            "sentence_logits": sentence_logits,
            "sentence_probs": sentence_probs,
            "sentence_labels": sentence_labels,
            "gates": gates,
            "hard_gates": hard_gates,
        }


### Auxiliary Task: Technique Classification
This cell defines the `TechniqueAuxiliaryTask`, a **multi-label classification** head that operates on the token features.
* **Mechanism:**
    * **Input:** Takes the same learned token embeddings used for the main BIO tagging task.
    * **Network:** A simple Feed-Forward Network (FFN) projects the embeddings to the number of technique classes.
    * **Loss:** Uses **Binary Cross Entropy (BCE)** with masking, treating the problem as multi-label classification (compatible with overlapping technique spans).

In [ ]:
class TechniqueAuxiliaryTask(nn.Module):
    """Token-level technique classification auxiliary branch (multi-label).

    Input: learned SI token feature vectors $e^{(si)}_i$.
    Processing: FFN.
    Output: multi-label prediction over propaganda techniques.
    """

    def __init__(
        self,
        *,
        input_dim: int,
        num_techniques: int,
        ffn_hidden_dim: int,
        output_dropout: float,
    ) -> None:
        super().__init__()
        if num_techniques <= 0:
            raise ValueError("num_techniques must be > 0")

        hidden = ffn_hidden_dim if ffn_hidden_dim > 0 else input_dim
        self.num_techniques = int(num_techniques)
        self.ffn = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(output_dropout),
            nn.Linear(hidden, self.num_techniques),
        )

    def forward(
        self,
        token_features: torch.Tensor,  # [B, L, D]
        *,
        targets: Optional[torch.Tensor] = None,  # [B, L, C]
        mask: Optional[torch.Tensor] = None,  # [B, L]
    ) -> Dict[str, torch.Tensor]:
        logits = self.ffn(token_features)  # [B, L, C]
        out: Dict[str, torch.Tensor] = {"tech_logits": logits}

        if targets is None:
            out["tech_loss"] = torch.tensor(0.0, device=logits.device)
            return out

        target = targets.float()
        if mask is None:
            mask = torch.ones(target.shape[:2], device=logits.device, dtype=torch.float)
        else:
            mask = mask.float()

        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, target, reduction="none")
        token_bce = bce.mean(dim=-1)  # [B, L]
        masked = token_bce * mask
        denom = mask.sum().clamp_min(1.0)
        out["tech_loss"] = masked.sum() / denom
        return out



### Main Model Architecture: BERT-BiLSTM-CRF
This cell defines the core `BertLstmCrf` class, which integrates the backbone, auxiliary tasks, and decoding logic into a unified **Multi-Task Learning (MTL)** framework.

* **Backbone & Context:**
    * **Encoder:** BERT (with `ScalarMix` for layer fusion) $\rightarrow$ BiLSTM (for sequence context).
    * **Feature Fusion:** Concatenates PLM embeddings with linguistic features (POS, NER, Dependency) and lexicon flags before the BiLSTM.
* **Multi-Task Objectives:**
    1.  **Span Identification (Primary):** A **CRF layer** decodes the BIO tag sequence.
    2.  **Sentence Gating (Auxiliary):** Uses `SentenceAuxiliaryTask` to classify sentences as propaganda/non-propaganda. This acts as a **hard gate**, zeroing out the token loss for non-propaganda sentences to reduce false positives.
    3.  **Technique Classification (Auxiliary):** Uses `TechniqueAuxiliaryTask` to predict specific techniques (e.g., "Loaded Language") from the token embeddings.
* **Loss Calculation:** Computes a weighted sum of `sentence_loss`, `token_loss` (masked by gates), and `tech_loss` (masked by gates).
* **Subword Handling:** Includes `mean_pool_subtokens` to align BERT's sub-word tokens with word-level linguistic labels and features.

In [ ]:

class BertLstmCrf(nn.Module):
    def __init__(
        self,
        bert_model: AutoModel,
        label_list: Sequence[str],
        pad_token_label_id: int,
        rnn_layers: int = 0,
        rnn_dropout: float = 0.1,
        output_dropout: float = 0.1,
        gate_enabled: bool = False,
        gate_function: str = "sigmoid",
        # Auxiliary task loss weights
        sentence_aux_weight: float = 1.0,  # Weight for sentence-level BCE loss
        token_loss_weight: float = 1.0,    # Weight for token-level CRF loss
        pos_vocab_size: int = 0,
        ner_vocab_size: int = 0,
        dep_vocab_size: int = 0,
        pos_embedding_dim: int = 50,
        ner_embedding_dim: int = 50,
        dep_embedding_dim: int = 50,
        lexicon_feature_dim: int = 0,
        use_scalar_mix: bool = True,
        rnn_hidden_size: int = 0,
        ffn_hidden_dim: int = 0,
        # Technique classification auxiliary task
        num_technique_classes: int = 0,
        tech_loss_weight: float = 0.5,
        # Mean pooling over subtokens
        use_mean_pooling: bool = False,
    ) -> None:
        super().__init__()
        self.bert_encoder = bert_model
        print(f'Rnn layers: {rnn_layers}')
        self.label_list = list(label_list)
        self.num_labels = len(label_list)
        self.pad_token_label_id = pad_token_label_id
        self.hidden_size = bert_model.config.hidden_size
        self.gate_enabled = gate_enabled
        
        # Loss weights for multi-task learning
        self.sentence_aux_weight = float(sentence_aux_weight)
        self.token_loss_weight = float(token_loss_weight)
        self.last_sentence_loss = 0.0
        
        self.use_scalar_mix = use_scalar_mix

        self.dropout = nn.Dropout(output_dropout)

        # Aux embeddings
        self.pos_vocab_size = pos_vocab_size
        self.ner_vocab_size = ner_vocab_size
        self.dep_vocab_size = dep_vocab_size
        self.lexicon_feature_dim = lexicon_feature_dim
        
        if pos_vocab_size > 0:
            self.pos_embedding = nn.Embedding(pos_vocab_size, pos_embedding_dim, padding_idx=0)
        else:
            self.pos_embedding = None
            pos_embedding_dim = 0
            
        if ner_vocab_size > 0:
            self.ner_embedding = nn.Embedding(ner_vocab_size, ner_embedding_dim, padding_idx=0)
        else:
            self.ner_embedding = None
            ner_embedding_dim = 0

        if dep_vocab_size > 0:
            self.dep_embedding = nn.Embedding(dep_vocab_size, dep_embedding_dim, padding_idx=0)
        else:
            self.dep_embedding = None
            dep_embedding_dim = 0
        
        # Store embedding dims for sentence auxiliary task
        self._pos_embedding_dim = pos_embedding_dim
        self._ner_embedding_dim = ner_embedding_dim
        self._dep_embedding_dim = dep_embedding_dim
        
        # Number of PLM layers for scalar mix
        num_plm_layers = getattr(bert_model.config, "num_hidden_layers", 12) + 1
        self.num_plm_layers = num_plm_layers
        
        if gate_enabled:
            # New sentence auxiliary task with separate PLM attention and BiLSTM
            self.sentence_aux = SentenceAuxiliaryTask(
                hidden_size=self.hidden_size,
                num_plm_layers=num_plm_layers,
                pos_embedding_dim=pos_embedding_dim,
                ner_embedding_dim=ner_embedding_dim,
                dep_embedding_dim=dep_embedding_dim,
                lexicon_feature_dim=self.lexicon_feature_dim,
                rnn_layers=1,  # Single layer BiLSTM for sentence task
                rnn_hidden_size=256,
                rnn_dropout=rnn_dropout,
                output_dropout=output_dropout,
                structural_feature_dim=4,  # length + upper/middle/lower
            )
        else:
            self.sentence_aux = None

        if self.use_scalar_mix:
             # Token-level scalar mix (separate from sentence-level)
             self.scalar_mix = ScalarMix(num_plm_layers)
        else:
             self.scalar_mix = None

        self.o_label_index = int(self.label_list.index("O")) if "O" in self.label_list else 0
        self.non_o_indices = [i for i, l in enumerate(self.label_list) if l != "O"]

        self.rnn_layers = rnn_layers
        self.embedding_dim = (
            self.hidden_size
            + pos_embedding_dim
            + ner_embedding_dim
            + dep_embedding_dim
            + self.lexicon_feature_dim
        )
        
        # Use provided rnn_hidden_size or default to embedding_dim // 2
        if rnn_hidden_size > 0:
            self.hidden_dim = rnn_hidden_size
        else:
            self.hidden_dim = max(1, self.embedding_dim // 2)

        if rnn_layers > 0:
            self.lstm = nn.LSTM(
                self.embedding_dim,
                self.hidden_dim,
                num_layers=rnn_layers,
                bidirectional=True,
                batch_first=True,
                dropout=rnn_dropout if rnn_layers > 1 else 0.0,
            )
            classifier_input = self.hidden_dim * 2
        else:
            self.lstm = None
            classifier_input = self.embedding_dim

        # FFN Implementation: Linear -> ReLU -> Dropout -> Linear
        # The paper specifies: "We apply a feed forward network (FFN) with one hidden layer ... before classification"
        # Use provided ffn_hidden_dim or default to classifier_input
        if ffn_hidden_dim > 0:
            effective_ffn_dim = ffn_hidden_dim
        else:
            effective_ffn_dim = classifier_input
        
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input, effective_ffn_dim),
            nn.ReLU(),
            nn.Dropout(output_dropout),
            nn.Linear(effective_ffn_dim, self.num_labels)
        )

        crf_labels = {idx: ("O" if label == "O" else label.split("-", 1)[0]) for idx, label in enumerate(self.label_list)}
        constraints = allowed_transitions("BIO", crf_labels)
        self.crf = ConditionalRandomField(self.num_labels, constraints, include_start_end_transitions=True)
        
        # NEW: Mean pooling flag
        self.use_mean_pooling = use_mean_pooling
        
        # NEW: Technique classification auxiliary task (multi-label, separate branch)
        self.num_technique_classes = int(num_technique_classes)
        self.tech_loss_weight = float(tech_loss_weight)
        self.last_tech_loss = 0.0
        
        if self.num_technique_classes > 0:
            # This branch consumes the learned SI token vectors (pre-BiLSTM).
            self.tech_aux = TechniqueAuxiliaryTask(
                input_dim=self.embedding_dim,
                num_techniques=self.num_technique_classes,
                ffn_hidden_dim=effective_ffn_dim,
                output_dropout=output_dropout,
            )
        else:
            self.tech_aux = None
    
    def mean_pool_subtokens(
        self, 
        subtoken_embeddings: torch.Tensor, 
        word_ids: torch.Tensor,
        max_words: Optional[int] = None
    ) -> torch.Tensor:
        """
        Mean pool subtoken embeddings to word-level embeddings.
        
        Args:
            subtoken_embeddings: [B, seq_len, hidden_dim] subtoken representations
            word_ids: [B, seq_len] mapping each subtoken to word index (-1 for special tokens)
            max_words: Optional maximum number of words (inferred from word_ids if not provided)
        
        Returns:
            word_embeddings: [B, max_words, hidden_dim] word-level representations
        """
        B, seq_len, hidden_dim = subtoken_embeddings.shape
        device = subtoken_embeddings.device
        
        if max_words is None:
            # Infer max words from word_ids (ignore -1)
            max_words = int((word_ids.max() + 1).item())
        
        if max_words == 0:
            # Fallback: just return the subtoken embeddings as-is
            return subtoken_embeddings
        
        # Initialize output and count tensors
        word_embeddings = torch.zeros(B, max_words, hidden_dim, device=device)
        word_counts = torch.zeros(B, max_words, device=device)
        
        # Create mask for valid word_ids (not -1)
        valid_mask = word_ids >= 0  # [B, seq_len]
        
        for b in range(B):
            for s in range(seq_len):
                w = word_ids[b, s].item()
                if w >= 0 and w < max_words:
                    word_embeddings[b, w] += subtoken_embeddings[b, s]
                    word_counts[b, w] += 1
        
        # Avoid division by zero
        word_counts = word_counts.clamp_min(1.0)
        word_embeddings = word_embeddings / word_counts.unsqueeze(-1)
        
        return word_embeddings

    def _mean_pool_feature_tensor(
        self,
        feature_tensor: torch.Tensor,
        word_ids: torch.Tensor,
    ) -> torch.Tensor:
        """Pool arbitrary [B, seq_len, dim] features to word level using mean."""
        if feature_tensor is None:
            return None
        B, seq_len, dim = feature_tensor.shape
        device = feature_tensor.device
        max_words = int((word_ids.max() + 1).item()) if word_ids.numel() > 0 else 0
        if max_words == 0:
            return feature_tensor
        pooled = torch.zeros(B, max_words, dim, device=device)
        counts = torch.zeros(B, max_words, device=device)
        for b in range(B):
            for s in range(seq_len):
                w = word_ids[b, s].item()
                if w >= 0 and w < max_words:
                    pooled[b, w] += feature_tensor[b, s]
                    counts[b, w] += 1
        counts = counts.clamp_min(1.0)
        pooled = pooled / counts.unsqueeze(-1)
        return pooled

    def forward(self, input_ids, attention_mask, pos_ids=None, ner_ids=None, labels=None,
                 sentence_lengths=None, sentence_positions=None, 
                 word_ids=None, tech_multi_labels=None, tech_label_mask=None,
                 dep_ids=None, lexicon_features=None):
        """
        Forward pass with optional sentence auxiliary task and technique classification.
        
        When sentence_aux is enabled:
        - Non-propaganda sentences: loss = sentence_classification_loss only
        - Propaganda sentences: loss = sentence_loss + lambda * token_level_BIO_loss
        
        Args:
            sentence_lengths: [B] sentence lengths for structural features
            sentence_positions: [B, 3] one-hot position encoding (upper/middle/lower)
            word_ids: [B, seq_len] subtoken-to-word mapping (-1 for special tokens)
            tech_multi_labels: [B, seq_len, num_techniques] multi-hot labels
            tech_label_mask: [B, seq_len] 1 for supervised positions
            dep_ids: [B, seq_len] dependency relation ids
            lexicon_features: [B, seq_len, lexicon_feature_dim] binary/float lexicon flags
        """
        # Always get all hidden states for sentence auxiliary task
        outputs = self.bert_encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states if hasattr(outputs, "hidden_states") else outputs[2]
        
        # Token-level PLM representation
        if self.use_scalar_mix:
            sequence_output = self.scalar_mix(hidden_states)
        else:
            sequence_output = outputs[0]
            
        sequence_output = self.dropout(sequence_output)

        # Build subtoken-level feature tensors for sentence auxiliary task.
        pos_embed_sentence = self.pos_embedding(pos_ids) if (self.pos_embedding is not None and pos_ids is not None) else None
        ner_embed_sentence = self.ner_embedding(ner_ids) if (self.ner_embedding is not None and ner_ids is not None) else None
        dep_embed_sentence = self.dep_embedding(dep_ids) if (self.dep_embedding is not None and dep_ids is not None) else None

        lex_sentence = None
        if self.lexicon_feature_dim > 0:
            if lexicon_features is None:
                B, L, _ = sequence_output.shape
                lex_sentence = torch.zeros(B, L, self.lexicon_feature_dim, device=sequence_output.device)
            else:
                lex_sentence = lexicon_features
        

        if self.use_mean_pooling and word_ids is not None:
            # Mean pool PLM embeddings to word level
            pooled_output = self.mean_pool_subtokens(sequence_output, word_ids)
                
            sequence_output = pooled_output
            
            # Adjust POS/NER/DEP to word level if needed
            if self.pos_embedding is not None and pos_ids is not None:
                # Extract first subtoken POS for each word
                word_pos = self._get_word_level_features(pos_ids, word_ids)
                pos_embed = self.pos_embedding(word_pos)
                sequence_output = torch.cat([sequence_output, pos_embed], dim=-1)
            
            if self.ner_embedding is not None and ner_ids is not None:
                word_ner = self._get_word_level_features(ner_ids, word_ids)
                ner_embed = self.ner_embedding(word_ner)
                sequence_output = torch.cat([sequence_output, ner_embed], dim=-1)

            if self.dep_embedding is not None:
                if dep_ids is not None:
                    word_dep = self._get_word_level_features(dep_ids, word_ids)
                else:
                    # Use padding index 0 which outputs zeros due to padding_idx=0
                    B, W, _ = sequence_output.shape
                    word_dep = torch.zeros(B, W, dtype=torch.long, device=sequence_output.device)
                dep_embed = self.dep_embedding(word_dep)
                sequence_output = torch.cat([sequence_output, dep_embed], dim=-1)

            if self.lexicon_feature_dim > 0:
                if lexicon_features is None:
                    # pad zeros to keep input dim stable
                    B, W, _ = sequence_output.shape
                    lexicon_features = torch.zeros(B, W, self.lexicon_feature_dim, device=sequence_output.device)
                else:
                    lexicon_features = self._mean_pool_feature_tensor(lexicon_features, word_ids)
                sequence_output = torch.cat([sequence_output, lexicon_features], dim=-1)
        else:
            # Standard subtoken-level processing
            pos_embed = None
            ner_embed = None
            dep_embed = None
            features_to_concat = [sequence_output]
            
            if self.pos_embedding is not None and pos_ids is not None:
                pos_embed = self.pos_embedding(pos_ids)
                features_to_concat.append(pos_embed)
                
            if self.ner_embedding is not None and ner_ids is not None:
                ner_embed = self.ner_embedding(ner_ids)
                features_to_concat.append(ner_embed)

            if self.dep_embedding is not None:
                if dep_ids is not None:
                    dep_embed = self.dep_embedding(dep_ids)
                else:
                    # Use padding index 0 which outputs zeros due to padding_idx=0
                    B, L, _ = sequence_output.shape
                    dep_ids_zeros = torch.zeros(B, L, dtype=torch.long, device=sequence_output.device)
                    dep_embed = self.dep_embedding(dep_ids_zeros)
                features_to_concat.append(dep_embed)

            if self.lexicon_feature_dim > 0:
                if lexicon_features is None:
                    B, L, _ = sequence_output.shape
                    lexicon_features = torch.zeros(B, L, self.lexicon_feature_dim, device=sequence_output.device)
                features_to_concat.append(lexicon_features)
                
            if len(features_to_concat) > 1:
                sequence_output = torch.cat(features_to_concat, dim=-1)

        # Technique classification aux branch uses the learned SI token vectors pre-BiLSTM.
        tech_loss = torch.tensor(0.0, device=sequence_output.device)
        if self.tech_aux is not None and tech_multi_labels is not None:
            aux_targets = tech_multi_labels
            aux_mask = tech_label_mask

            # When using mean pooling, convert subtoken-level targets to word-level.
            if self.use_mean_pooling and word_ids is not None:
                B = word_ids.size(0)
                max_words = int((word_ids.max() + 1).item()) if word_ids.numel() > 0 else 0
                if max_words > 0:
                    word_targets = torch.zeros(B, max_words, self.num_technique_classes, device=sequence_output.device)
                    word_mask = torch.zeros(B, max_words, device=sequence_output.device)
                    for b in range(B):
                        seen_words = set()
                        for s in range(word_ids.size(1)):
                            w = int(word_ids[b, s].item())
                            if w >= 0 and w < max_words and w not in seen_words:
                                seen_words.add(w)
                                word_targets[b, w] = aux_targets[b, s]
                                if aux_mask is not None:
                                    word_mask[b, w] = aux_mask[b, s]
                                else:
                                    word_mask[b, w] = 1.0
                    aux_targets = word_targets
                    aux_mask = word_mask

            tech_out = self.tech_aux(sequence_output, targets=aux_targets, mask=aux_mask)
            tech_loss = tech_out["tech_loss"]
            self.last_tech_loss = float(tech_loss.detach().item())

        if self.lstm is not None:
            sequence_output, _ = self.lstm(sequence_output)

        logits = self.classifier(sequence_output)

        # =================== FAST PATH WHEN SENTENCE AUX IS OFF ===================
        if self.sentence_aux is None:
            # When using mean pooling, labels and mask need to be at word-level
            if self.use_mean_pooling and word_ids is not None:
                word_labels = self._get_word_level_features(labels, word_ids)
                # Create word-level mask (1 where we have valid words)
                max_words = logits.size(1)
                word_mask = torch.zeros(labels.size(0), max_words, device=labels.device, dtype=attention_mask.dtype)
                for b in range(labels.size(0)):
                    for w in range(max_words):
                        # A word position is valid if any subtoken maps to it
                        if (word_ids[b] == w).any():
                            word_mask[b, w] = 1
                base_loss, _, predicted_tags = self._decode_without_gating(logits, word_labels, word_mask)
            else:
                base_loss, _, predicted_tags = self._decode_without_gating(logits, labels, attention_mask)
            # Add technique loss if applicable
            total_loss = base_loss + self.tech_loss_weight * tech_loss
            return total_loss, logits, predicted_tags
        # ===========================================================================

        # Run sentence auxiliary task with separate PLM attention
        aux_out = self.sentence_aux(
            plm_hidden_states=hidden_states,
            pos_embeddings=pos_embed_sentence,
            ner_embeddings=ner_embed_sentence,
            dep_embeddings=dep_embed_sentence,
            lexicon_features=lex_sentence,
            attention_mask=attention_mask,
            labels=labels,
            pad_token_label_id=self.pad_token_label_id,
            o_label_index=self.o_label_index,
            sentence_lengths=sentence_lengths,
            sentence_positions=sentence_positions,
        )

        if labels is None:
            raise ValueError("labels tensor is required to align sub-tokens in this condensed pipeline")

        clear_logits, clear_labels, clear_mask = self._clear_subtokens(logits, labels, attention_mask)

        # Viterbi decoding for predictions
        best_paths = self.crf.viterbi_tags(clear_logits, clear_mask.long(), top_k=1)
        predicted_tags = [path_scores[0][0] for path_scores in best_paths]

        # Compute sentence classification loss (BCE)
        sentence_loss = torch.tensor(0.0, device=logits.device)
        if aux_out["sentence_labels"] is not None:
            bce = nn.BCEWithLogitsLoss()
            sentence_loss = bce(aux_out["sentence_logits"], aux_out["sentence_labels"])
        
        # Hard gates: 0 for non-propaganda, 1 for propaganda sentences
        hard_gates = aux_out["hard_gates"]  # [B]
        
        # Compute per-sample CRF log likelihood
        # We need to compute loss per sample to apply different weights
        batch_size = clear_logits.size(0)
        token_losses = []
        
        for i in range(batch_size):
            sample_logits = clear_logits[i:i+1]
            sample_labels = clear_labels[i:i+1]
            sample_mask = clear_mask[i:i+1]
            
            if sample_mask.sum() == 0:
                # Skip empty sequences
                token_losses.append(torch.tensor(0.0, device=logits.device))
                continue
                
            sample_ll = self.crf(sample_logits, sample_labels, sample_mask.long())
            token_losses.append(-sample_ll)
        
        token_losses = torch.stack(token_losses)  # [B]
        
        # Apply gating to BOTH token loss and tech loss:
        # - Non-propaganda (gate=0): ONLY sentence_loss is used
        # - Propaganda (gate=1): sentence_loss + token_loss + tech_loss
        gated_token_losses = token_losses * hard_gates
        
        # Mean over batch, but only count propaganda sentences for token loss
        num_prop_sentences = hard_gates.sum().clamp_min(1.0)
        mean_token_loss = gated_token_losses.sum() / num_prop_sentences
        
        # Gate the technique loss too - only compute for propaganda sentences
        # If all sentences are non-propaganda, tech_loss contribution is 0
        has_propaganda = hard_gates.sum() > 0
        gated_tech_loss = tech_loss if has_propaganda else torch.tensor(0.0, device=logits.device)
        
        # Final loss computation with configurable weights:
        # Non-propaganda: loss = sentence_weight * sentence_loss
        # Propaganda: loss = sentence_weight * sentence_loss + token_weight * token_loss + tech_weight * tech_loss
        loss = (self.sentence_aux_weight * sentence_loss + 
                self.token_loss_weight * mean_token_loss + 
                self.tech_loss_weight * gated_tech_loss)
        
        # Store for logging
        self.last_sentence_loss = float(sentence_loss.detach().item())

        return loss, logits, predicted_tags
    
    def _get_word_level_features(self, feature_ids: torch.Tensor, word_ids: torch.Tensor) -> torch.Tensor:
        """
        Extract word-level features from subtoken-level features.
        Takes the first subtoken's feature for each word.
        
        Args:
            feature_ids: [B, seq_len] subtoken-level feature IDs (e.g., POS/NER)
            word_ids: [B, seq_len] subtoken-to-word mapping
            
        Returns:
            word_features: [B, max_words] word-level feature IDs
        """
        B, seq_len = feature_ids.shape
        device = feature_ids.device
        max_words = int((word_ids.max() + 1).item())
        
        if max_words == 0:
            return feature_ids
        
        word_features = torch.zeros(B, max_words, dtype=feature_ids.dtype, device=device)
        
        for b in range(B):
            seen_words = set()
            for s in range(seq_len):
                w = word_ids[b, s].item()
                if w >= 0 and w < max_words and w not in seen_words:
                    word_features[b, w] = feature_ids[b, s]
                    seen_words.add(w)
        
        return word_features

    def _decode_without_gating(self, logits, labels, attention_mask):
        if labels is None:
            raise ValueError("labels tensor is required to align sub-tokens in this condensed pipeline")

        clear_logits, clear_labels, clear_mask = self._clear_subtokens(logits, labels, attention_mask)
        best_paths = self.crf.viterbi_tags(clear_logits, clear_mask.long(), top_k=1)
        predicted_tags = [p[0][0] for p in best_paths]

        log_likelihood = self.crf(clear_logits, clear_labels, clear_mask.long())
        # Normalize by total number of valid tokens to get per-token loss
        num_tokens = clear_mask.sum().clamp_min(1.0)
        loss = -log_likelihood / num_tokens

        return loss, logits, predicted_tags

    def _clear_subtokens(self, logits, labels, mask):
        batch_size, seq_length, num_labels = logits.size()
        clear_logits = logits.new_zeros((batch_size, seq_length, num_labels))
        clear_labels = labels.new_full((batch_size, seq_length), fill_value=self.pad_token_label_id)
        clear_mask = mask.new_zeros((batch_size, seq_length))

        for row in range(batch_size):
            valid_positions = labels[row] != self.pad_token_label_id
            trimmed_logits = logits[row][valid_positions]
            trimmed_labels = labels[row][valid_positions]
            length = trimmed_labels.size(0)
            if length == 0:
                continue
            clear_logits[row, :length] = trimmed_logits
            clear_labels[row, :length] = trimmed_labels
            clear_mask[row, :length] = 1

        clear_labels = clear_labels.masked_fill(clear_mask == 0, 0)
        return clear_logits, clear_labels, clear_mask

### Scoring & Post-Processing Utilities
This cell implements the official evaluation metrics and heuristics to refine model predictions:

* **Evaluation (`compute_precision_recall_f1`)**: Calculates Precision, Recall, and F1 scores based on **character-level overlap** between predicted spans and gold annotations.
* **Span Adjustment (`_adjust_span_to_words`)**: "Snaps" predicted character offsets to the nearest word boundaries, correcting cases where the model might cut off half a word.
* **Post-Processing (`postprocess_spans`)**:
    * **Filters** insignificant spans (e.g., single stopwords like "the", "is").
    * **Merges** adjacent spans that are separated only by punctuation or single connector words.
* **Aggregation (`aggregate_article_spans`)**: Reconstructs full article-level predictions by stitching together the flattened list of token labels output by the model batches.

In [ ]:
def load_span_annotations(filename: str) -> Dict[str, List[Tuple[int, int]]]:
    annotations: Dict[str, List[Tuple[int, int]]] = {}
    with open(filename, "r", encoding="utf-8") as handle:
        for line in handle:
            article_id, start, end = line.rstrip("\n").split("\t")
            annotations.setdefault(article_id, []).append((int(start), int(end)))
    return annotations


def convert_to_position_sets(annotations: Dict[str, List[Tuple[int, int]]]) -> Dict[str, List[set]]:
    converted: Dict[str, List[set]] = {}
    for article_id, spans in annotations.items():
        converted[article_id] = [set(range(start, end)) for start, end in spans if end > start]
    return converted


def compute_precision_recall_f1(
    predicted: Dict[str, List[Tuple[int, int]]],
    gold: Dict[str, List[Tuple[int, int]]],
) -> Dict[str, float]:
    pred_sets = convert_to_position_sets(predicted)
    gold_sets = convert_to_position_sets(gold)
    prec_den = sum(len(spans) for spans in pred_sets.values())
    rec_den = sum(len(spans) for spans in gold_sets.values())
    prec_num = 0.0
    rec_num = 0.0
    for article_id, pred_spans in pred_sets.items():
        gold_spans = gold_sets.get(article_id, [])
        for pred_span in pred_spans:
            pred_len = len(pred_span)
            if pred_len == 0:
                continue
            for gold_span in gold_spans:
                intersection = len(pred_span & gold_span)
                if intersection:
                    prec_num += intersection / pred_len
        for gold_span in gold_spans:
            gold_len = len(gold_span)
            if gold_len == 0:
                continue
            for pred_span in pred_spans:
                intersection = len(pred_span & gold_span)
                if intersection:
                    rec_num += intersection / gold_len
    precision = prec_num / prec_den if prec_den else 0.0
    recall = rec_num / rec_den if rec_den else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision and recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


# --- END: Logic distilled from task-SI_scorer.py ---


# --- START: Span post-processing (word boundary alignment) ---
def _is_word_char(ch: str) -> bool:
    return ch.isalnum() or ch == "_"


def _is_span_boundary_char(ch: str) -> bool:
    """Check if character should be included in span boundaries.
    
    Includes alphanumeric, underscore, and punctuation that can be part of spans.
    """
    # Keep these punctuation marks in spans:
    # - Quotes and apostrophes: ' ' " " "
    # - Hyphens and dashes: - – —
    # - End-of-statement: ! ? .
    KEEP_PUNCT = {"'", "'", '"', '"', '"', '-', '–', '—', '!', '?', '.'}
    return ch.isalnum() or ch == "_" or ch in KEEP_PUNCT


def _adjust_span_to_words(text: str, s: int, e: int, keep_punct: bool = False) -> Optional[Tuple[int, int]]:
    n = len(text)
    s = max(0, min(s, n))
    e = max(0, min(e, n))
    if e <= s:
        return None
    
    # Choose which function to use for boundary detection
    is_boundary_char = _is_span_boundary_char if keep_punct else _is_word_char
    
    # Trim leading non-boundary chars (only whitespace)
    while s < e and text[s].isspace():
        s += 1
    # Trim trailing non-boundary chars (only whitespace)
    while e > s and text[e - 1].isspace():
        e -= 1
    
    if not keep_punct:
        # Original behavior: also trim leading/trailing punctuation
        while s < e and not _is_word_char(text[s]):
            s += 1
        while e > s and not _is_word_char(text[e - 1]):
            e -= 1
    
    if e <= s:
        return None
    # Expand backward if start is mid-word
    if s > 0 and _is_word_char(text[s]) and _is_word_char(text[s - 1]):
        while s > 0 and _is_word_char(text[s - 1]):
            s -= 1
    # Expand forward if end is mid-word (e is exclusive)
    if e < n and _is_word_char(text[e - 1]) and _is_word_char(text[e]):
        while e < n and _is_word_char(text[e]):
            e += 1
    if e <= s:
        return None
    return s, e


def postprocess_spans(
    spans: Dict[str, List[Tuple[int, int]]],
    article_texts: Dict[str, str],
    keep_punct: bool = True,  # Keep punctuation in spans by default
) -> Dict[str, List[Tuple[int, int]]]:
    # Words to remove if a span contains only a single one of these (case-insensitive)
    SINGLE_WORD_STOP = {
        # Articles & Determiners
        "a", "an", "the", "this", "that", "these", "those", "some", "any", "all", "each", "every",
        # Pronouns
        "i", "me", "my", "mine", "you", "your", "yours", "we", "us", "our", "ours",
        "he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their", "theirs",
        "who", "whom", "whose", "which",
        # Prepositions
        "of", "in", "to", "for", "with", "on", "at", "from", "by", "about", "as",
        "into", "like", "through", "after", "over", "between", "out", "against",
        "during", "without", "before", "under", "around", "among", "via",
        # Conjunctions
        "and", "but", "or", "nor", "so", "yet", "if", "because", "while", "when", "where", "although", "unless",
        # Verbs (Be/Have/Do/Modals)
        "is", "am", "are", "was", "were", "be", "been", "being","its",
        "have", "has", "had", "do", "does", "did",
        "will", "would", "shall", "should", "can", "could", "may", "might", "must",
        # Adverbs/Negation/Misc
        "not", "no", "yes", "just", "very", "really", "now", "then", "here", "there", "also", "too",
    }
    adjusted: Dict[str, List[Tuple[int, int]]] = {}
    for aid, span_list in spans.items():
        text = article_texts.get(aid)
        if text is None:
            adjusted[aid] = list(span_list)
            continue
        new_spans: List[Tuple[int, int]] = []
        for s, e in span_list:
            adj = _adjust_span_to_words(text, s, e, keep_punct=keep_punct)
            if adj is not None:
                # Filter out single-word spans that match stop list
                ss, ee = adj
                frag = text[ss:ee]
                words = [w.lower() for w in __import__("re").findall(r"[A-Za-z0-9_]+", frag)]
                if len(words) == 1 and words[0] in SINGLE_WORD_STOP:
                    continue
                new_spans.append(adj)
        # De-duplicate and sort
        unique_sorted = sorted(set(new_spans))

        # Merge with rules: allow merge across gaps that are only punctuation/whitespace
        # or a single connector word (letters/digits/_), mirroring pipeline behavior
        merged: List[Tuple[int, int]] = []
        for s, e in unique_sorted:
            if merged:
                ps, pe = merged[-1]
                if pe <= s:
                    gap = text[pe:s]
                    # Rule 1: gap is only whitespace or punctuation (limited set)
                    if gap and all(ch in string.whitespace or ch in ',:;.!?"\'()-[]' for ch in gap):
                        merged[-1] = (ps, e)
                        continue
                    # Rule 2: gap is a single connector word (only word chars), optional spaces around
                    stripped = gap.strip()
                    if stripped and all(_is_word_char(ch) for ch in stripped):
                        merged[-1] = (ps, e)
                        continue
            merged.append((s, e))

        adjusted[aid] = merged
    return adjusted

# --- END: Span post-processing (word boundary alignment) ---
def aggregate_article_spans(
    sentence_level_labels: Sequence[Sequence[str]],
    article_order: Sequence[str],
    token_cache: Dict[str, List[TokenSpan]],
) -> Dict[str, List[Tuple[int, int]]]:
    flat_labels: List[str] = [label for sequence in sentence_level_labels for label in sequence]
    pointer = 0
    spans: Dict[str, List[Tuple[int, int]]] = {}
    for article_id in article_order:
        tokens = token_cache.get(article_id, [])
        token_count = len(tokens)
        if token_count == 0:
            spans[article_id] = []
            continue
        remaining_labels = len(flat_labels) - pointer
        take = min(token_count, max(0, remaining_labels))
        article_labels = flat_labels[pointer : pointer + take]
        pointer += take
        if take < token_count:
            article_labels.extend(["O"] * (token_count - take))
        spans[article_id] = labels_to_spans(tokens[: len(article_labels)], article_labels)
    if pointer != len(flat_labels):  # pragma: no cover - diagnostic aid
        print("Unused label predictions detected: %d", len(flat_labels) - pointer)
    return spans


### Training & Inference Pipeline
* **`build_dataset`**: Converts raw BIO files into a PyTorch `TensorDataset`, packing all multi-task features (Input IDs, POS/NER IDs, Technique labels, Lexicon flags).
* **`train_model`**: The central training loop. It features:
    * **Encoder Freezing**: Freezes BERT layers for the first few epochs to stabilize the randomly initialized BiLSTM/CRF layers.
    * **Span-Level Evaluation**: Converts token predictions to spans during validation to use the official competition metric (Span F1) for **Early Stopping**.
    * **Gradient Accumulation**: Supports large effective batch sizes.
    * **Visualization**: Plots loss curves and PR curves (via WandB).
* **`predict_on_articles`**: An end-to-end inference utility. It accepts raw `.txt` files, runs the spaCy preprocessing, generates BIO tags, loads the trained model, and outputs the final submission format.

In [ ]:


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class DatasetBundle:
    examples: List[InputExample]
    features: List[InputFeatures]
    dataset: TensorDataset
    article_ids: List[str]


def build_dataset(
    file_path: str,
    mode: str,
    article_ids: List[str],
    tokenizer,
    label_list: Sequence[str],
    max_seq_length: int,
    pad_token_label_id: int,
    pos_label_map: Optional[Dict[str, int]] = None,
    ner_label_map: Optional[Dict[str, int]] = None,
    dep_label_map: Optional[Dict[str, int]] = None,
    token_cache: Optional[Dict[str, List["TokenSpan"]]] = None,
    technique_spans_by_article: Optional[Dict[str, List[Tuple[int, int, str]]]] = None,
    technique_labels: Optional[List[str]] = None,
) -> DatasetBundle:
    examples = read_examples_from_file(file_path, mode)

    tech_label_map: Optional[Dict[str, int]] = None
    if technique_labels is not None:
        tech_label_map = get_multilabel_technique_map(technique_labels)

    if technique_spans_by_article is not None and token_cache is not None:
        attach_technique_labels_to_examples(
            examples,
            article_ids=article_ids,
            token_cache=token_cache,
            technique_spans_by_article=technique_spans_by_article,
        )
    features = convert_examples_to_features(
        examples, 
        label_list, 
        max_seq_length, 
        tokenizer, 
        pad_token_label_id,
        pos_label_map=pos_label_map,
        ner_label_map=ner_label_map,
        dep_label_map=dep_label_map,
        tech_label_map=tech_label_map,
    )
    all_input_ids = torch.tensor([f.input_ids for f in features], dtype=torch.long)
    all_attention_mask = torch.tensor([f.attention_mask for f in features], dtype=torch.long)
    all_label_ids = torch.tensor([f.label_ids for f in features], dtype=torch.long)
    all_pos_ids = torch.tensor([f.pos_ids for f in features], dtype=torch.long)
    all_ner_ids = torch.tensor([f.ner_ids for f in features], dtype=torch.long)
    all_dep_ids = torch.tensor([f.dep_ids if f.dep_ids else [0] * max_seq_length for f in features], dtype=torch.long)
    all_word_ids = torch.tensor([f.word_ids for f in features], dtype=torch.long)
    tech_dim = len(tech_label_map) if tech_label_map else 0
    all_tech_multi_labels = torch.tensor(
        [
            f.tech_multi_labels if f.tech_multi_labels is not None else [[0] * tech_dim] * max_seq_length
            for f in features
        ],
        dtype=torch.float,
    )
    all_tech_label_mask = torch.tensor(
        [
            f.tech_label_mask if f.tech_label_mask is not None else [0] * max_seq_length
            for f in features
        ],
        dtype=torch.float,
    )
    all_lexicon_features = torch.tensor(
        [
            f.lexicon_features if f.lexicon_features is not None else [[0]] * max_seq_length
            for f in features
        ],
        dtype=torch.float,
    )

    dataset = TensorDataset(
        all_input_ids,
        all_attention_mask,
        all_label_ids,
        all_pos_ids,
        all_ner_ids,
        all_dep_ids,
        all_word_ids,
        all_lexicon_features,
        all_tech_multi_labels,
        all_tech_label_mask,
    )
    return DatasetBundle(examples=examples, features=features, dataset=dataset, article_ids=list(article_ids))

def prepare_bio_files(
    work_dir: str,
    nlp,
    *,
    mode: str,
    articles: Sequence[str],
    article_ids: Sequence[str],
    labels_path: Optional[str] = None,
    token_cache: Optional[Dict[str, List[TokenSpan]]] = None,
    filename: Optional[str] = None,
) -> Tuple[str, List[str]]:
    """Materialise a single BIO file for the requested split."""
    os.makedirs(work_dir, exist_ok=True)
    if filename is None:
        filename = f"{mode}.bio"
    output_path = os.path.join(work_dir, filename)

    articles_content = dict(zip(article_ids, articles))

    if token_cache is not None:
        for article_id, text in articles_content.items():
            token_cache[article_id] = tokenize_like_bio(text, nlp)

    normalized_mode = mode.lower()
    labeled_modes = {"train", "dev"}
    prediction_modes = {"test", "prediction"}

    if normalized_mode in labeled_modes:
        if labels_path is None:
            raise ValueError("labels_path is required when mode is train or dev")
        article_id_labels, spans = read_predictions_from_file(labels_path)
        grouped_dict = group_spans_by_article_ids(zip(article_id_labels, spans))
        for article_id in article_ids:
            grouped_dict.setdefault(article_id, [])
        ordered = [(article_id, grouped_dict.get(article_id, [])) for article_id in article_ids]
        create_bio_labeled(output_path, ordered, articles_content, nlp)
    elif normalized_mode in prediction_modes:
        create_bio_unlabeled(output_path, article_ids, articles, nlp)
    else:
        raise ValueError(f"Unsupported prepare_bio_files mode: {mode}")

    return output_path, list(article_ids)
def evaluate_model(
    model: BertLstmCrf,
    dataloader: DataLoader,
    label_list: Sequence[str],
    pad_token_label_id: int,
) -> Tuple[Dict[str, float], List[List[str]], Optional[Tuple[np.ndarray, np.ndarray]]]:
    model.eval()
    preds: List[List[str]] = []
    gold: List[List[str]] = []
    losses: List[float] = []
    
    # Storage for PR curve data
    all_probs: List[float] = [] # Probability of PROP class
    all_labels: List[int] = []  # 1 if PROP, 0 if O
    
    with torch.no_grad():
        for batch in dataloader:
            batch = tuple(t.to(DEVICE) for t in batch)
            # batch: 0=input, 1=mask, 2=labels, ...
            loss, logits, predicted = model(
                input_ids=batch[0], 
                attention_mask=batch[1], 
                labels=batch[2],
                pos_ids=batch[3],
                ner_ids=batch[4],
                dep_ids=batch[5],
                word_ids=batch[6],
                lexicon_features=batch[7],
                tech_multi_labels=batch[8],
                tech_label_mask=batch[9],
            )
            if loss is not None:
                losses.append(loss.item())
            
            # --- Collect probabilities for PR Curve ---
            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1) # [B, L, C]
            
            for idx, sequence in enumerate(predicted):
                label_ids = [lid for lid in batch[2][idx].tolist() if lid != pad_token_label_id]
                seq_len = len(label_ids)
                
                gold_labels = [label_list[lid] for lid in label_ids]
                pred_labels = [label_list[tag_id] for tag_id in sequence[: seq_len]]
                gold.append(gold_labels)
                preds.append(pred_labels)
                
                
                # Identify 'O' index
                o_idx = -1
                if "O" in label_list:
                    o_idx = label_list.index("O")
                
                if o_idx >= 0:
                    # p(Propaganda) = 1 - p(O)
                    # p_prop = 1.0 - probs[idx, :seq_len, o_idx].cpu().numpy()                    
                    token_probs = probs[idx, :seq_len].cpu().numpy()
                    token_gold = label_ids
                    
                    for t_i in range(seq_len):
                        # Binary ground truth
                        is_prop = 1 if label_list[token_gold[t_i]] != "O" else 0
                        # Probability of being propaganda
                        p_prop = 1.0 - token_probs[t_i, o_idx]
                        
                        all_probs.append(p_prop)
                        all_labels.append(is_prop)

    precision = precision_score(gold, preds) if gold else 0.0
    recall = recall_score(gold, preds) if gold else 0.0
    f1 = f1_score(gold, preds) if gold else 0.0
    metrics = {
        "loss": float(np.mean(losses)) if losses else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    
    pr_data = (np.array(all_labels), np.array(all_probs)) if all_labels else None
    return metrics, preds, pr_data


def predict_model(
    model: BertLstmCrf,
    bundle: DatasetBundle,
    label_list: Sequence[str],
    pad_token_label_id: int,
    batch_size: int,
    num_workers: int = 0,
) -> List[List[str]]:
    dataloader = DataLoader(
        bundle.dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=str(DEVICE) != "cpu" and not TPU_AVAILABLE
    )
    model.eval()
    predictions: List[List[str]] = []
    with torch.no_grad():
        for batch in dataloader:
            batch = tuple(t.to(DEVICE) for t in batch)
            _, _, predicted = model(
                input_ids=batch[0], 
                attention_mask=batch[1], 
                labels=batch[2],
                pos_ids=batch[3],
                ner_ids=batch[4],
                dep_ids=batch[5],
                word_ids=batch[6],
                lexicon_features=batch[7],
                tech_multi_labels=batch[8],
                tech_label_mask=batch[9],
            )
            for idx, sequence in enumerate(predicted):
                label_ids = [lid for lid in batch[2][idx].tolist() if lid != pad_token_label_id]
                predictions.append([label_list[tag_id] for tag_id in sequence[: len(label_ids)]])
    return predictions

def train_model(
    model: BertLstmCrf,
    train_bundle: DatasetBundle,
    dev_bundle: DatasetBundle,
    label_list: Sequence[str],
    pad_token_label_id: int,
    train_batch_size: int,
    eval_batch_size: int,
    learning_rate: float,
    weight_decay: float,
    num_epochs: int,
    warmup_steps: int,
    max_grad_norm: float,
    patience: int = 2,
    accum_steps: int = 1,  
    *,
    freeze_encoder_epochs: int = 2,
    save_checkpoint_epochs: int = 0,
    checkpoint_dir: Optional[str] = None,
    resume_from_checkpoint: Optional[str] = None,
    dev_gold_spans: Optional[Dict[str, List[Tuple[int, int]]]] = None,
    token_cache: Optional[Dict[str, List[TokenSpan]]] = None,
    dev_article_texts: Optional[Dict[str, str]] = None,
    apply_postprocess: bool = True,
    num_workers: int = 0,
) -> Dict[str, float]:
    
    # Dataloader creation (single-process mode, no distributed sampler needed)
    train_loader = DataLoader(
        train_bundle.dataset, 
        batch_size=train_batch_size, 
        shuffle=True,
        num_workers=num_workers,
        pin_memory=str(DEVICE) != "cpu" and not TPU_AVAILABLE
    )
    dev_loader = DataLoader(
        dev_bundle.dataset, 
        batch_size=eval_batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=str(DEVICE) != "cpu" and not TPU_AVAILABLE
    )


    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    # Total optimizer steps (ceil to include remainder accumulation group)
    import math
    total_steps = max(1, math.ceil(len(train_loader) * num_epochs / max(1, accum_steps)))

    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=warmup_steps, 
        num_training_steps=total_steps
    )

    best_state = copy.deepcopy(model.state_dict())
    best_metrics = {"f1": 0.0, "precision": 0.0, "recall": 0.0}

    optimizer.zero_grad()
    wait = 0
    global_step = 0
    start_epoch = 1
    
    # Track losses for plotting (step-level for smoother curves)
    step_train_losses = []  # (global_step, loss) tuples
    step_dev_losses = []    # (global_step, loss) tuples
    log_every_n_steps = 100  # Log train loss every N steps
    running_loss = 0.0
    running_count = 0
    
    # Resume from checkpoint if provided
    if resume_from_checkpoint and os.path.exists(resume_from_checkpoint):
        print(f"Resuming from checkpoint: {resume_from_checkpoint}")
        checkpoint = torch.load(resume_from_checkpoint, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        global_step = checkpoint.get("global_step", 0)
        best_metrics = checkpoint.get("best_metrics", best_metrics)
        best_state = checkpoint.get("best_state", best_state)
        wait = checkpoint.get("wait", 0)
        print(f"Resumed from epoch {checkpoint['epoch']}, best F1: {best_metrics['f1']:.4f}")
    
    # Freeze PLM (BERT encoder) for the first N epochs
    if freeze_encoder_epochs > 0:
        for param in model.bert_encoder.parameters():
            param.requires_grad = False
        print(f"PLM (bert_encoder) frozen for first {freeze_encoder_epochs} epochs")

    for epoch in range(start_epoch, num_epochs + 1):
        # Unfreeze PLM after the specified number of frozen epochs
        if freeze_encoder_epochs > 0 and epoch == freeze_encoder_epochs + 1:
            for param in model.bert_encoder.parameters():
                param.requires_grad = True
            print(f"PLM (bert_encoder) unfrozen at epoch {epoch}")
        
        model.train()

        epoch_loss = 0.0
        
        iteration_loader = train_loader
        if TPU_AVAILABLE and DEVICE.type == 'xla':
             # Use ParallelLoader for TPU performance
             iteration_loader = pl.ParallelLoader(train_loader, [DEVICE]).per_device_loader(DEVICE)

        
        remainder = len(train_loader) % accum_steps if accum_steps > 0 else 0
        
        
        for step, batch in enumerate(tqdm(iteration_loader, desc=f"Epoch {epoch}/{num_epochs}", mininterval=60, disable=TPU_AVAILABLE)):
            if not (TPU_AVAILABLE and DEVICE.type == 'xla'):
                batch = tuple(t.to(DEVICE) for t in batch)
            
            loss, _, _ = model(
                input_ids=batch[0],
                attention_mask=batch[1],
                labels=batch[2],
                pos_ids=batch[3],
                ner_ids=batch[4],
                dep_ids=batch[5],
                word_ids=batch[6],
                lexicon_features=batch[7],
                tech_multi_labels=batch[8],
                tech_label_mask=batch[9],
            )
            if loss is None:
                continue

            # Determine actual group size (last group may be smaller)
            if accum_steps <= 0:
                group_size = 1
            else:
                # If we are in the final remainder region and remainder != 0, use remainder; else use accum_steps
                if remainder and step >= len(train_loader) - remainder:
                    group_size = remainder
                else:
                    group_size = accum_steps

            scaled_loss = loss / group_size
            scaled_loss.backward()
            current_loss = scaled_loss.item()
            epoch_loss += current_loss
            
            # Track running loss for step-level logging
            running_loss += current_loss
            running_count += 1

            should_step = False
            if accum_steps <= 1:
                should_step = True
            else:
                # Regular accumulation boundary
                if (step + 1) % accum_steps == 0:
                    should_step = True
                # Final partial group
                if (step + 1) == len(train_loader):
                    should_step = True

            if should_step:
                if TPU_AVAILABLE and DEVICE.type == 'xla':
                    xm.optimizer_step(optimizer)
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                
                # Log training loss every N steps
                if global_step % log_every_n_steps == 0 and running_count > 0:
                    avg_loss = running_loss / running_count
                    step_train_losses.append((global_step, avg_loss))
                    running_loss = 0.0
                    running_count = 0
        
        # Reduce loss for logging
        if TPU_AVAILABLE and DEVICE.type == 'xla':
             # xm.master_print or similar
             pass
        
        print(
            f"Epoch {epoch} training loss: {epoch_loss / max(1, len(train_loader)):.4f}"
        )


        metrics, dev_sentence_predictions, dev_pr_data = evaluate_model(model, dev_loader, label_list, pad_token_label_id)
        
        # Track dev loss at current global_step
        step_dev_losses.append((global_step, metrics['loss']))

        print(
            f"Dev token-level metrics after epoch {epoch} - F1: {metrics['f1']:.4f} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}"
        )

        # Compute span-level metrics (required for best model selection)
        span_metrics = {"f1": 0.0, "precision": 0.0, "recall": 0.0}
        if dev_gold_spans is not None and token_cache is not None:
            try:
                dev_spans = aggregate_article_spans(
                    dev_sentence_predictions,
                    dev_bundle.article_ids,
                    token_cache,
                )
                if apply_postprocess and dev_article_texts is not None:
                    dev_spans = postprocess_spans(dev_spans, dev_article_texts)

                span_metrics = compute_precision_recall_f1(dev_spans, dev_gold_spans)
                print(
                    f"Dev span-level metrics after epoch {epoch} - F1: {span_metrics['f1']:.4f} | Precision: {span_metrics['precision']:.4f} | Recall: {span_metrics['recall']:.4f}"
                )
                if wandb and wandb.run:
                    wandb.log({
                        "epoch": epoch,
                        "loss/train": epoch_loss / max(1, len(train_loader)),
                        "loss/dev": metrics['loss'],
                        "metrics/token_f1_dev": metrics['f1'],
                        "metrics/token_precision_dev": metrics['precision'],
                        "metrics/token_recall_dev": metrics['recall'],
                        "metrics/span_f1_dev": span_metrics['f1'],
                        "metrics/span_precision_dev": span_metrics['precision'],
                        "metrics/span_recall_dev": span_metrics['recall'],
                    })
                    
                    if dev_pr_data is not None:
                        y_true, y_scores = dev_pr_data
                
                        
                        # Downsample for speed if huge
                        if len(y_true) > 10000:
                            indices = np.random.choice(len(y_true), 10000, replace=False)
                            y_true = y_true[indices]
                            y_scores = y_scores[indices]
                            

                        try:
                            probs_2d = np.vstack([1 - y_scores, y_scores]).T
                            wandb.log({"pr_curve_epoch": wandb.plot.pr_curve(y_true, probs_2d, labels=["O", "PROP"])})
                        except Exception as e:
                            print(f"Failed to log PR curve: {e}")
            except Exception:
                LOGGER.exception("Span-level evaluation failed for epoch %d", epoch)

        # Early stopping logic based on span-level F1 (primary metric)
        if span_metrics["f1"] > best_metrics["f1"]:
            best_metrics = span_metrics
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
            print(f"New best span F1: {span_metrics['f1']:.4f} at epoch {epoch}")
        else:
            wait += 1
            if wait == patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break
        
        # Save checkpoint every N epochs
        if save_checkpoint_epochs > 0 and epoch % save_checkpoint_epochs == 0:
            ckpt_dir = checkpoint_dir if checkpoint_dir else "."
            ckpt_path = os.path.join(ckpt_dir, f"checkpoint_epoch_{epoch}.pt")
            checkpoint_data = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "global_step": global_step,
                "best_metrics": best_metrics,
                "best_state": best_state,
                "wait": wait,
            }
            torch.save(checkpoint_data, ckpt_path)
            print(f"Checkpoint saved to {ckpt_path}")

    model.load_state_dict(best_state)
    
    if step_train_losses or step_dev_losses:
        try:
            plt.figure(figsize=(12, 6))
            
            if step_train_losses:
                train_steps, train_losses = zip(*step_train_losses)
                plt.plot(train_steps, train_losses, 'b-', label='Train Loss', linewidth=1.5, alpha=0.8)
            
            # Plot dev loss (logged at end of each epoch)
            if step_dev_losses:
                dev_steps, dev_losses = zip(*step_dev_losses)
                plt.plot(dev_steps, dev_losses, 'r-o', label='Dev Loss', linewidth=2, markersize=8)
            
            plt.xlabel('Training Steps', fontsize=12)
            plt.ylabel('Loss (normalized per token)', fontsize=12)
            plt.title('Training vs Validation Loss', fontsize=14)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            
            # Save to checkpoint directory or current directory
            loss_plot_path = os.path.join(checkpoint_dir if checkpoint_dir else ".", "loss_curve.png")
            plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"Loss curve saved to: {loss_plot_path}")
        except Exception as e:
            print(f"Failed to plot loss curve: {e}")
    
    return best_metrics


def predict_on_articles(
    model_path: str,
    prediction_articles_path: str,
    *,
    model_name: str = "roberta-base",
    max_seq_length: int = 256,
    batch_size: int = 24,
    nlp=None,
    prediction_labels: str = "prediction_tokens.bio",
    prediction_submission_path: str = "prediction_submission.tsv",
    apply_postprocess: bool = True,
    pos_ner_maps_path: Optional[str] = None,
    pos_label_map: Optional[Dict[str, int]] = None,
    ner_label_map: Optional[Dict[str, int]] = None,
):
    """Run inference on raw articles and emit BIO predictions plus submission spans.
    
    Args:
        model_path: Path to the saved model checkpoint.
        prediction_articles_path: Directory containing .txt article files.
        model_name: Name of the HuggingFace model (must match training).
        max_seq_length: Max sequence length for tokenization.
        batch_size: Batch size for inference.
        nlp: spaCy model instance (required for POS/NER extraction).
        prediction_labels: Output path for BIO predictions.
        prediction_submission_path: Output path for submission file.
        apply_postprocess: Whether to apply span post-processing.
        pos_ner_maps_path: Path to JSON file containing pos_label_map and ner_label_map.
        pos_label_map, ner_label_map: Alternatively, pass the maps directly.
    """
    import json
    
    # Load maps from file if path provided
    if pos_ner_maps_path is not None and (pos_label_map is None or ner_label_map is None):
        with open(pos_ner_maps_path, "r", encoding="utf-8") as f:
            maps_data = json.load(f)
        pos_label_map = maps_data.get("pos_label_map", {})
        ner_label_map = maps_data.get("ner_label_map", {})
        print("Loaded POS/NER maps from %s", pos_ner_maps_path)
    
    if pos_label_map is None:
        pos_label_map = {}
    if ner_label_map is None:
        ner_label_map = {}
    
    prediction_labels_path = Path(prediction_labels)
    prediction_submission_path_obj = Path(prediction_submission_path)

    work_dir_path = (
        prediction_labels_path.parent
        if prediction_labels_path.parent != Path("")
        else Path.cwd()
    )
    work_dir_path.mkdir(parents=True, exist_ok=True)
    if prediction_submission_path_obj.parent != Path(""):
        prediction_submission_path_obj.parent.mkdir(parents=True, exist_ok=True)

    articles_path = Path(prediction_articles_path)
    if not articles_path.exists():
        raise FileNotFoundError(
            f"Prediction articles path does not exist: {prediction_articles_path}"
        )

    prediction_articles = []
    prediction_article_ids = []

    file_list = sorted(
        p for p in articles_path.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    )

    for article_file in file_list:
        with article_file.open("r", encoding="utf-8") as handle:
            prediction_articles.append(handle.read())

        stem = article_file.stem
        if stem.startswith("article"):
            prediction_article_ids.append(stem[len("article"):])
        else:
            prediction_article_ids.append(stem)

    if not prediction_articles:
        raise ValueError(f"No .txt articles found in {prediction_articles_path}")

    token_cache = {}
    bio_path, article_order = prepare_bio_files(
        str(work_dir_path),
        nlp,
        mode="prediction",
        articles=prediction_articles,
        article_ids=prediction_article_ids,
        token_cache=token_cache,
        filename="prediction.bio",
    )

    label_list = get_labels()
    pad_token_label_id = torch.nn.CrossEntropyLoss().ignore_index

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name)

    enable_tech_aux = True
    technique_labels = get_technique_labels(None)
    num_technique_classes = max(0, len(technique_labels) - (1 if technique_labels and technique_labels[0] == "O" else 0))

    pos_vocab_size = len(pos_label_map) + 1 if pos_label_map else 0
    ner_vocab_size = len(ner_label_map) + 1 if ner_label_map else 0
    dep_vocab_size = 0  # Provide >0 if dependency ids are supplied for inference
    dep_embedding_dim = 50
    lexicon_feature_dim = 1  # Binary Hitler-keyword flag

    model = BertLstmCrf(
        base_model,
        label_list,
        pad_token_label_id,
        rnn_layers=1,
        rnn_dropout=0.1,
        output_dropout=getattr(base_model.config, "hidden_dropout_prob", 0.1),
        pos_vocab_size=pos_vocab_size,
        ner_vocab_size=ner_vocab_size,
        pos_embedding_dim=25,
        ner_embedding_dim=25,
        dep_vocab_size=dep_vocab_size,
        dep_embedding_dim=dep_embedding_dim,
        lexicon_feature_dim=lexicon_feature_dim,
        num_technique_classes=(num_technique_classes if enable_tech_aux else 0),
        tech_loss_weight=0.5,
    ).to(DEVICE)

    state_dict = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state_dict)

    dataset_bundle = build_dataset(
        bio_path,
        "prediction",
        article_order,
        tokenizer,
        label_list,
        max_seq_length,
        pad_token_label_id,
        pos_label_map=pos_label_map,
        ner_label_map=ner_label_map,
    )

    predictions = predict_model(
        model, dataset_bundle, label_list, pad_token_label_id, batch_size
    )
    spans = aggregate_article_spans(predictions, article_order, token_cache)
    if apply_postprocess:
        article_texts_map = dict(zip(prediction_article_ids, prediction_articles))
        spans = postprocess_spans(spans, article_texts_map)

    write_token_predictions(bio_path, predictions, prediction_labels_path)
    write_submission_file(spans, prediction_submission_path_obj)

    return {
        "bio": Path(bio_path),
        "token_predictions": prediction_labels_path,
        "submission": prediction_submission_path_obj,
    }


### Main Execution & Experiment Configuration
This block is the entry point for the script, orchestrating the end-to-end training and evaluation pipeline.

* **Hyperparameter Configuration**: Manually sets training parameters (e.g., `batch_size`, `learning_rate`, `epochs`), model dimensions (e.g., `rnn_hidden_size`, `pos_embedding_dim`), and auxiliary task weights.
* **Environment Setup**: Handles path resolution (compatible with Kaggle, Colab, and local environments), device selection (TPU/GPU/CPU), and random seeding for reproducibility.
* **WandB Integration**: Initializes Weights & Biases for experiment tracking if API keys are available.
* **Data Preparation**:
    * Loads raw article texts and ground truth labels.
    * Initializes **spaCy** and generates BIO-formatted files for Train, Dev, and Test splits via `prepare_bio_files`.
    * Builds vocabulary maps for POS, NER, and Dependency tags from the training data.
* **Model Initialization**: Instantiates the `BertLstmCrf` model with the configured backbone (`roberta-large`) and auxiliary task heads (Technique Classification).
* **Execution Flow**:
    1.  **Build Datasets**: Converts BIO data into tensor-ready `DatasetBundle` objects.
    2.  **Train**: Runs the training loop (`train_model`) with early stopping and checkpointing.
    3.  **Evaluate**: Runs inference on Dev and Test sets.
    4.  **Post-Process**: Aggregates token predictions into spans, applies word-boundary snapping, and computes final F1 scores.
    5.  **Save**: Exports predictions to submission files (`.tsv`) and saves the final model weights.

In [ ]:


if __name__ == "__main__":
    import argparse
    import os
    import json
    
    parser = None 

    # Manual Configuration
    num_workers = 1         
    use_tpu = True          
    
    train_batch_size = 16
    eval_batch_size = 64
    learning_rate = 2e-5
    weight_decay = 0.0
    warmup_steps = 500
    max_grad_norm = 1.0
    num_epochs = 15
    freeze_encoder_epochs = 2  
    seed = 1024
    
    # Model Hyperparameters (User Configured)
    pos_embedding_dim = 50    
    ner_embedding_dim = 50    
    dep_embedding_dim = 50    
    lexicon_feature_dim = 1   
    rnn_hidden_size = 600     
    rnn_layers = 2            
    rnn_dropout = 0.1         
    output_dropout = 0.1      
    ffn_hidden_dim = 200      
    
    gate_function = "sigmoid"
    
    # Auxiliary Task Loss Weights (Configurable)
    sentence_aux_weight = 1.0   # Weight for sentence-level auxiliary task (BCE loss)
    token_loss_weight = 1.0     # Weight for token-level BIO/CRF loss
    tech_aux_weight = 0.5       # Weight for technique classification auxiliary task
    enable_tech_aux = True      # Enable multi-label technique auxiliary branch (14 labels)
    
    patience = 5
    accum_steps = 1
    
    # Checkpoint settings
    save_checkpoint_epochs = 0   # Save checkpoint every N epochs (0 to disable)
    resume_from_checkpoint = None  # Path to checkpoint file to resume from (e.g., "checkpoint_epoch_5.pt")

    # Device Setup
    # Device Setup
    # Do NOT initialize XLA device here when using spawn!
    if not (TPU_AVAILABLE and (use_tpu or os.environ.get("TPU_NAME"))):
        DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {DEVICE}")
    else:
        print("TPU enabled. Device will be initialized in spawned processes.")

    model_name = "roberta-large"
    
    max_seq_length = 256


    data_root = Path("/kaggle/input/propaganda-dataset/datasets") if os.path.exists("/kaggle/input/propaganda-dataset/datasets") else Path("/content/drive/MyDrive/ColabNotebooks/datasets")
    work_dir = Path("/kaggle/working/") if os.path.exists("/kaggle/working/") else Path("/content/drive/MyDrive/ColabNotebooks/output")
    output_dir = work_dir
    
    # Check if we are local
    if not data_root.exists():
        # Fallback for local testing if path logic above fails
        data_root = Path(r"D:\Project3\project\datasets")
    
    # (Optional) technique names file is not required locally; `get_technique_labels()` has defaults.
    techniques_file = data_root.parent / "propaganda-techniques-names.txt"
    if not techniques_file.exists():
        techniques_file = data_root / "propaganda-techniques-names.txt"
    
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    set_seed(seed)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading data from {data_root}")
    # Ensure directories exist
    if (data_root / "train-articles").exists():
        train_articles, train_ids, _ = load_data(str(data_root / "train-articles"), str(techniques_file))
        dev_articles, dev_ids, _ = load_data(str(data_root / "dev-articles"), str(techniques_file))
        test_articles, test_ids, _ = load_data(str(data_root / "test-articles"), str(techniques_file))

        train_labels_path = data_root / "train-task1-SI.labels"
        dev_labels_path = data_root / "dev-task1-SI.labels"
        test_labels_path = data_root / "test-task1-SI.labels"

        train_tc_labels_path = data_root / "train-task2-TC.labels"
        dev_tc_labels_path = data_root / "dev-task2-TC.labels"
        test_tc_labels_path = data_root / "test-task2-TC.labels"

        dev_prediction_labels = output_dir / "dev_predictions.bio"
        test_prediction_labels = output_dir / "test_predictions.bio"

        dev_submission_path = output_dir / "dev_submission.tsv"
        test_submission_path = output_dir / "test_submission.tsv"

        # --- Step 2: Initialize spaCy ---
        print("Initializing spaCy...")
        nlp = get_configured_spacy("en_core_web_sm")

        # --- Step 3: Create BIO labels (Run once in main process) ---
        token_cache: Dict[str, List[TokenSpan]] = {}

        train_bio, train_order = prepare_bio_files(
            str(work_dir), nlp, mode="train", articles=train_articles, article_ids=train_ids,
            labels_path=str(train_labels_path), token_cache=token_cache, filename="train.bio",
        )
        dev_bio, dev_order = prepare_bio_files(
            str(work_dir), nlp, mode="dev", articles=dev_articles, article_ids=dev_ids,
            labels_path=str(dev_labels_path), token_cache=token_cache, filename="dev.bio",
        )
        test_bio, test_order = prepare_bio_files(
            str(work_dir), nlp, mode="test", articles=test_articles, article_ids=test_ids,
            token_cache=token_cache, filename="test.bio",
        )
        
        dev_gold_annotations = load_span_annotations(str(dev_labels_path))
        dev_texts_map = dict(zip(dev_ids, dev_articles))
        test_texts_map = dict(zip(test_ids, test_articles))

        technique_labels = get_technique_labels(str(techniques_file) if techniques_file.exists() else None)
        train_tc_spans = read_technique_spans(str(train_tc_labels_path)) if train_tc_labels_path.exists() else {}
        dev_tc_spans = read_technique_spans(str(dev_tc_labels_path)) if dev_tc_labels_path.exists() else {}
        test_tc_spans = read_technique_spans(str(test_tc_labels_path)) if test_tc_labels_path.exists() else {}


        # Pack flags for mp_fn
        flags = {
            "model_name": model_name,
            "max_seq_length": max_seq_length,
            "train_bio": train_bio,
            "dev_bio": dev_bio,
            "test_bio": test_bio,
            "train_order": train_order,
            "dev_order": dev_order,
            "test_order": test_order,
            "num_workers": num_workers,
            "train_batch_size": train_batch_size,
            "eval_batch_size": eval_batch_size,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "num_epochs": num_epochs,
            "warmup_steps": warmup_steps,
            "max_grad_norm": max_grad_norm,
            "patience": patience,
            "accum_steps": accum_steps,
            "token_cache": token_cache,
            "dev_gold_annotations": dev_gold_annotations,
            "dev_texts_map": dev_texts_map,
            "test_texts_map": test_texts_map,
            "output_dir": str(output_dir),
            "dev_prediction_labels": str(dev_prediction_labels),
            "test_prediction_labels": str(test_prediction_labels),
            "dev_submission_path": str(dev_submission_path),
            "test_submission_path": str(test_submission_path),
            "test_labels_path": str(test_labels_path),
            
            # Hyperparams
            "pos_embedding_dim": pos_embedding_dim,
            "ner_embedding_dim": ner_embedding_dim,
            "dep_embedding_dim": dep_embedding_dim,
            "lexicon_feature_dim": lexicon_feature_dim,
            "rnn_hidden_size": rnn_hidden_size,
            "rnn_layers": rnn_layers,
            "rnn_dropout": rnn_dropout,
            "output_dropout": output_dropout,
            "ffn_hidden_dim": ffn_hidden_dim,
            "freeze_encoder_epochs": freeze_encoder_epochs,
            "save_checkpoint_epochs": save_checkpoint_epochs,
            "resume_from_checkpoint": resume_from_checkpoint,
            "seed": seed
        }

        if TPU_AVAILABLE and use_tpu:
            DEVICE = torch_xla.device()
            print(f"Using TPU device: {DEVICE} (single-process mode)")
        else:
            DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            print(f"Using device: {DEVICE}")
        
        # Re-seed inside process
        set_seed(flags["seed"])
        
        # --- Step 4: Create data loaders ---
        label_list = get_labels()
        pad_token_label_id = torch.nn.CrossEntropyLoss().ignore_index
        tokenizer = AutoTokenizer.from_pretrained(flags["model_name"])
        base_model = AutoModel.from_pretrained(flags["model_name"])

        print("Building POS/NER/DEP maps from training data...")
        
        # Read examples (redundant across procs but necessary for complete objects)
        train_examples_temp = read_examples_from_file(flags["train_bio"], "train")
        pos_label_map, ner_label_map, dep_label_map = get_pos_ner_dep_maps(train_examples_temp)
        
        print(f"  POS labels: {len(pos_label_map)}, NER labels: {len(ner_label_map)}, DEP labels: {len(dep_label_map)}")

        model = BertLstmCrf(
            base_model,
            label_list,
            pad_token_label_id,
            rnn_layers=flags["rnn_layers"],
            rnn_dropout=flags["rnn_dropout"],
            output_dropout=getattr(base_model.config, "hidden_dropout_prob", flags["output_dropout"]),
            pos_vocab_size=len(pos_label_map) + 1, 
            ner_vocab_size=len(ner_label_map) + 1,
            dep_vocab_size=len(dep_label_map) + 1,  # Use actual dep vocab size from data
            pos_embedding_dim=flags["pos_embedding_dim"],
            ner_embedding_dim=flags["ner_embedding_dim"],
            dep_embedding_dim=flags["dep_embedding_dim"],
            lexicon_feature_dim=flags["lexicon_feature_dim"],
            use_scalar_mix=True,
            rnn_hidden_size=flags["rnn_hidden_size"],
            ffn_hidden_dim=flags["ffn_hidden_dim"],
            use_mean_pooling=True,
            num_technique_classes=(
                max(0, len(technique_labels) - (1 if technique_labels and technique_labels[0] == "O" else 0))
                if enable_tech_aux
                else 0
            ),
            tech_loss_weight=tech_aux_weight,
        ).to(DEVICE)

        train_bundle = build_dataset(
            flags["train_bio"], "train", flags["train_order"], tokenizer, label_list, flags["max_seq_length"],
            pad_token_label_id, pos_label_map=pos_label_map, ner_label_map=ner_label_map, dep_label_map=dep_label_map,
            token_cache=flags["token_cache"],
            technique_spans_by_article=train_tc_spans,
            technique_labels=technique_labels,
        )
        dev_bundle = build_dataset(
            flags["dev_bio"], "dev", flags["dev_order"], tokenizer, label_list, flags["max_seq_length"],
            pad_token_label_id, pos_label_map=pos_label_map, ner_label_map=ner_label_map, dep_label_map=dep_label_map,
            token_cache=flags["token_cache"],
            technique_spans_by_article=dev_tc_spans,
            technique_labels=technique_labels,
        )
        test_bundle = build_dataset(
            flags["test_bio"], "test", flags["test_order"], tokenizer, label_list, flags["max_seq_length"],
            pad_token_label_id, pos_label_map=pos_label_map, ner_label_map=ner_label_map, dep_label_map=dep_label_map,
            token_cache=flags["token_cache"],
            technique_spans_by_article=test_tc_spans,
            technique_labels=technique_labels,
        )

        # --- Step 5: Training ---
        best_metrics = train_model(
            model, train_bundle, dev_bundle, label_list, pad_token_label_id,
            flags["train_batch_size"], flags["eval_batch_size"], flags["learning_rate"], flags["weight_decay"],
            flags["num_epochs"], flags["warmup_steps"], flags["max_grad_norm"], flags["patience"], flags["accum_steps"],
            freeze_encoder_epochs=flags["freeze_encoder_epochs"],
            save_checkpoint_epochs=flags["save_checkpoint_epochs"],
            checkpoint_dir=flags["output_dir"],
            resume_from_checkpoint=flags["resume_from_checkpoint"],
            dev_gold_spans=flags["dev_gold_annotations"], token_cache=flags["token_cache"],
            dev_article_texts=flags["dev_texts_map"], apply_postprocess=True,
            num_workers=flags["num_workers"]
        )
        
        print(f"Best dev F1: {best_metrics['f1']:.4f}")

        # --- Step 6: Evaluation ---
        dev_predictions = predict_model(model, dev_bundle, label_list, pad_token_label_id, flags["eval_batch_size"], num_workers=flags["num_workers"])
        test_predictions = predict_model(model, test_bundle, label_list, pad_token_label_id, flags["eval_batch_size"], num_workers=flags["num_workers"])

        # Single-process mode - output unconditionally
        dev_token_spans = aggregate_article_spans(dev_predictions, flags["dev_order"], flags["token_cache"])
        test_token_spans = aggregate_article_spans(test_predictions, flags["test_order"], flags["token_cache"])

        dev_token_spans = postprocess_spans(dev_token_spans, flags["dev_texts_map"])
        test_token_spans = postprocess_spans(test_token_spans, flags["test_texts_map"])
        
        write_token_predictions(flags["dev_bio"], dev_predictions, Path(flags["dev_prediction_labels"]))
        write_token_predictions(flags["test_bio"], test_predictions, Path(flags["test_prediction_labels"]))
        write_submission_file(dev_token_spans, Path(flags["dev_submission_path"]))
        write_submission_file(test_token_spans, Path(flags["test_submission_path"]))
        
        # Metrics
        dev_metrics = compute_precision_recall_f1(dev_token_spans, flags["dev_gold_annotations"])
        test_gold_annotations = load_span_annotations(flags["test_labels_path"])
        test_metrics = compute_precision_recall_f1(test_token_spans, test_gold_annotations)

        print(f"Dev submission metrics: {dev_metrics}")
        print(f"Test submission metrics: {test_metrics}")

        # --- Step 7: Save checkpoint ---
        torch.save(model.state_dict(), Path(flags["output_dir"]) / "best_model.pt")



    else:
        print(f"Data directories not found at {data_root}")
